In [1]:
# Installing Required Modules
!uv sync

Resolved 28 packages in 600ms
Prepared 3 packages in 476ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 24 packages in 2.17s
 + annotated-types==0.7.0
 + anyio==4.12.0
 + authlib==1.6.6
 + certifi==2026.1.4
 + cffi==2.0.0
 + cryptography==46.0.3
 + deprecation==2.1.0
 + grpcio==1.76.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.11
 + numpy==2.4.0
 + pandas==2.3.3
 + protobuf==6.33.2
 + pycparser==2.23
 + pydantic==2.12.5
 + pydantic-core==2.41.5
 + pytz==2025.2
 + typing-extensions==4.15.0
 + typing-inspection==0.4.2
 + tzdata==2025.3
 + validators==0.35.0
 + weaviate-client==4.19.2


### 向量資料庫連線和建立Collection

In [ ]:
import weaviate
import json
import pandas as pd 

client = weaviate.connect_to_local() 
client.is_ready() 
print("Weaviate (v4) 連線成功。")

Weaviate (v4) 連線成功。


In [145]:
import weaviate
from weaviate.classes.config import Property, DataType, Configure

client = weaviate.connect_to_local()

articles_collection = client.collections.create(
    name="Articles",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="abstractText", data_type=DataType.TEXT),
    ],

    # 向量化
    vectorizer_config=Configure.Vectorizer.text2vec_ollama(
        model="nomic-embed-text",
        api_endpoint="http://120.113.70.238:11434"  
    ),

    generative_config=Configure.Generative.ollama(
        model="gpt-oss:20b",
        api_endpoint="http://120.113.70.238:11434"
    )
)

articles_collection = client.collections.get("Articles")

print("Connected and collection ready!")

2026-01-26 14:31:13,683 INFO HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-01-26 14:31:13,904 INFO HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-01-26 14:31:14,247 INFO HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
d:\RAG\rag-ecosystem\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\user\AppData\Local\Temp\ipykernel_10744\286769946.py:4: ResourceWarning: unclosed <socket.socket fd=3880, family=23, type=1, proto=0, laddr=('::1', 59015, 0, 0), raddr=('::1', 8080, 0, 0)>
  client = weaviate.connect_to_local()
d:\RAG\rag-ecosystem\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collect

Connected and collection ready!


C:\Users\user\AppData\Local\Temp\ipykernel_10744\286769946.py:6: ResourceWarning: unclosed <socket.socket fd=3188, family=23, type=1, proto=0, laddr=('::1', 53069, 0, 0), raddr=('::1', 8080, 0, 0)>
  articles_collection = client.collections.create(


In [3]:
collections = client.collections.list_all()
print(collections)

{'FeedbackLoopDocuments': _CollectionConfigSimple(name='FeedbackLoopDocuments', description=None, generative_config=None, properties=[_Property(name='chunk_id', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='document_id', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='source_type', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, text_analyzer=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_co

In [23]:
client.collections.delete("FeedbackLoopDocuments")
print("已刪除collection")

已刪除collection


### 爬蟲文章

In [147]:
import requests
from bs4 import BeautifulSoup

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

target_classes = ["post-title", "post-header", "post-content"]
full_text = ""

for class_name in target_classes:
    elements = soup.find_all(class_=class_name)
    for element in elements:
        full_text += element.get_text(separator="\n") + "\n\n"

print(full_text)

# import bs4
# from langchain_community.document_loaders import WebBaseLoader

# # Initialize a web document loader with specific parsing instructions
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),  # URL of the blog post to load
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")  # Only parse specified HTML classes
#         )
#     ),
# )

# # Load the filtered content from the web page into documents
# docs = loader.load()


      LLM Powered Autonomous Agents
    




      LLM Powered Autonomous Agents
    


Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as 
AutoGPT
, 
GPT-Engineer
 and 
BabyAGI
, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.


Agent System Overview
#


In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:




Planning




Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.


Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby imp

In [148]:
def simple_text_splitter(text, chunk_size=1000, chunk_overlap=200):
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)
    return chunks

full_text = full_text
my_chunks = simple_text_splitter(full_text, chunk_size=1000, chunk_overlap=200)
print(my_chunks)

# from langchain.text_splitter import RecursiveCharacterTextSplitter

# # Create a text splitter to divide text into chunks of 1000 characters with 200-character overlap
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# # Split the loaded documents into smaller chunks
# splits = text_splitter.split_documents(docs)

['\n      LLM Powered Autonomous Agents\n    \n\n\n\n\n      LLM Powered Autonomous Agents\n    \n\n\nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as \nAutoGPT\n, \nGPT-Engineer\n and \nBabyAGI\n, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\n\n\nAgent System Overview\n#\n\n\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\n\n\n\nPlanning\n\n\n\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\n\n\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes 

In [149]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

batch_size = 10
objects = []

for index, text_chunk in enumerate(my_chunks):
    try:
        obj = {
            "title": f"Title {index+1}",
            "abstractText": text_chunk
        }
        objects.append(obj)

        if len(objects) >= batch_size:
            articles_collection.data.insert_many(objects)
            logging.info(f"Inserted {index + 1} rows")
            objects = []

    except Exception as e:
        logging.error(f"Error processing row {index}: {e}")

if objects:
    articles_collection.data.insert_many(objects)
    logging.info("Final batch inserted")

logging.info("All data inserted successfully!")

response = articles_collection.aggregate.over_all(total_count=True)
print(f"成功匯入資料總數: {response.total_count}")


# from langchain_community.vectorstores import Chroma
# from langchain_openai import OpenAIEmbeddings

# # Embed the text chunks and store them in a Chroma vector store for similarity search
# vectorstore = Chroma.from_documents(
#     documents=splits, 
#     embedding=OpenAIEmbeddings()  # Use OpenAI's embedding model to convert text into vectors
# )

2026-01-26 14:31:27,573 INFO Inserted 10 rows
2026-01-26 14:31:27,715 INFO Inserted 20 rows
2026-01-26 14:31:27,858 INFO Inserted 30 rows
2026-01-26 14:31:27,972 INFO Inserted 40 rows
2026-01-26 14:31:28,101 INFO Inserted 50 rows
2026-01-26 14:31:28,196 INFO Final batch inserted
2026-01-26 14:31:28,197 INFO All data inserted successfully!


成功匯入資料總數: 55


In [ ]:
# client.close()

<a id='part1-2'></a>
## Retrieval

The vector store is our library, and the **retriever** is our smart librarian. It takes a user’s query, embeds it, and then fetches the most semantically similar chunks from the vector store.

![Retrieval Phase (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*jtf1FoBGfpnDPTTu9N94Wg.png)

In [151]:
class Document:
    def __init__(self, page_content, metadata=None):
        self.page_content = page_content
        self.metadata = metadata or {}

class MyWeaviateRetriever:
    def __init__(self, client, collection_name="Articles"):
        self.client = client
        self.collection = client.collections.get(collection_name)

    def get_relevant_documents(self, query):
        try:
            # 直接呼叫 Weaviate v4 API 進行向量搜尋
            # Weaviate 會自動呼叫 Ollama 轉向量
            response = self.collection.query.near_text(
                query=query,
                limit=1
            )

            # 將結果轉換成 LangChain 風格的 Document 物件列表
            docs = []
            for obj in response.objects:
                content = obj.properties.get("abstractText", "")
                title = obj.properties.get("title", "")
       
                doc = Document(page_content=content, metadata={"title": title})
                docs.append(doc)
            
            return docs

        except Exception as e:
            print(f"fail: {e}")
            return []
        

# # Create a retriever from the vector store
# retriever = vectorstore.as_retriever()

In [152]:
retriever = MyWeaviateRetriever(client)
docs = retriever.get_relevant_documents("What is Task Decomposition?")
print(docs[0].page_content)

# # Retrieve relevant documents for a query
# docs = retriever.get_relevant_documents("What is Task Decomposition?")
# # Print the content of the first retrieved document
# print(docs[0].page_content)



GPT-Engineer
 is another project to create a whole repository of code given a task specified in natural language. The GPT-Engineer is instructed to think over a list of smaller components to build and ask for user input to clarify questions as needed.


Here are a sample conversation for task clarification sent to OpenAI ChatCompletion endpoint used by GPT-Engineer. The user inputs are wrapped in 
{{user input text}}
.


[
  {
    "role": "system",
    "content": "You will read instructions and not carry them out, only seek to clarify them.\nSpecifically you will first summarise a list of super short bullets of areas that need clarification.\nThen you will pick one clarifying question, and wait for an answer from the user.\n"
  },
  {
    "role": "user",
    "content": "We are writing {{a Super Mario game in python. MVC components split in separate files. Keyboard control.}}\n"
  },
  {
    "role": "assistant",
    "content": "Summary of areas that need clarification:\n1. Specifics o

C:\Users\user\AppData\Local\Temp\ipykernel_10744\1111859092.py:1: ResourceWarning: unclosed <socket.socket fd=576, family=23, type=1, proto=0, laddr=('::1', 52870, 0, 0), raddr=('::1', 8080, 0, 0)>
  retriever = MyWeaviateRetriever(client)


In [18]:
client.close()

<a id='part1-3'></a>
### RAG的標準運作流程 (LLM在回答問題之前，先去翻書找答案，再回覆)
![Generation Step (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*0K6ognTAEOJQmb6KDL9wBw.png)

##### 角色指令下達

In [153]:
prompt = """You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.
            Use three sentences maximum and keep the answer concise.

            Question: {question}
            Context: {context}
            Answer:
        """

##### 先去資料庫找跟問題最像的文章內容，整理好，再拿去給LLM看。

In [154]:
from openai import OpenAI

llm_client = OpenAI(
    base_url="http://120.113.70.238:11434/v1",
    api_key="ollama"
)

MODEL_NAME = "gpt-oss:20b"
articles_collection = client.collections.get("Articles")
def get_formatted_context(query, k=3):
    response = articles_collection.query.near_text(query=query, limit=k)
    texts = [obj.properties.get("abstractText", "") for obj in response.objects]
    return "\n\n".join(texts)


# from langchain_openai import ChatOpenAI

# # Initialize the LLM
# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


In [155]:
question = "What is Task Decomposition?"

context_text = get_formatted_context(question)

final_prompt = prompt.format(
    question=question,
    context=context_text
)

response = llm_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": final_prompt}],
    temperature=0
)

print(response.choices[0].message.content)


# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough

# # Helper function to format retrieved documents
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

# # Define the full RAG chain
# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )
# Ask a question using the RAG chain
# response = rag_chain.invoke("What is Task Decomposition?")
# print(response)

2026-01-26 14:31:49,523 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


Task Decomposition is the process of breaking a larger, complex task into a set of smaller, manageable sub‑tasks or components. In projects like GPT‑Engineer, the system first lists these sub‑components and then seeks clarification on each before proceeding. This approach helps clarify requirements, organize work, and guide the model in building a complete solution.


<a id='part2'></a>
# Advanced Query Transformations

![Query Transformation (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*FO2U9QA49kjn6OaBGZuq8A.png)

#####
第一步：查詢優化/翻譯
使用者的問題通常很爛（模糊、關鍵字不對），我們要用LLM幫改寫成電腦好懂的問題。

第二步：混合檢索
單純用向量(Vector)有時候不準，加入關鍵字搜尋

第三步：重排序
搜尋找回來的5筆資料，可能第1筆是向量很像但內容無關的，要用一個專門的「評分模型」重新打分數。

第四步：生成回答
把經過 翻譯 -> 混合搜尋 -> 重排序 後留下的黃金資料，餵給LLM寫答案

In [156]:
import weaviate
from weaviate.classes.config import Property, DataType, Configure

client = weaviate.connect_to_local()

articles_collection = client.collections.create(
    name="Articles_2",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="abstractText", data_type=DataType.TEXT),
    ],

    # 向量化：指向的 Ollama
    vectorizer_config=Configure.Vectorizer.text2vec_ollama(
        model="nomic-embed-text",
        api_endpoint="http://120.113.70.238:11434"  
    ),

    # 生成式設定
    generative_config=Configure.Generative.ollama(
        model="gpt-oss:20b",
        api_endpoint="http://120.113.70.238:11434"
    )
)

articles_collection = client.collections.get("Articles_2")

print("Connected and collection ready!")

2026-01-26 14:31:58,157 INFO HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-01-26 14:31:58,389 INFO HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-01-26 14:31:58,760 INFO HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
2026-01-26 14:31:59,036 INFO HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"


Connected and collection ready!


In [157]:
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

target_classes = ["post-content", "post-title", "post-header"]
full_text = ""

for class_name in target_classes:
    elements = soup.find_all(class_=class_name)
    for element in elements:
        full_text += element.get_text(separator="\n") + "\n\n"

print(full_text)


# # Load the blog post
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")
#         )
#     ),
# )
# blog_docs = loader.load()

Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as 
AutoGPT
, 
GPT-Engineer
 and 
BabyAGI
, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.


Agent System Overview
#


In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:




Planning




Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.


Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.






Memory




Short-term memory: I would consider all the in-context learning (See 
Prompt Engineering
) as utilizing short-term memo

In [158]:
def simple_text_splitter(text, chunk_size=300, chunk_overlap=50):
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)
    return chunks

full_text = full_text
my_chunks = simple_text_splitter(full_text, chunk_size=300, chunk_overlap=50)
print(my_chunks)


# text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     chunk_size=300, 
#     chunk_overlap=50
# )
# splits = text_splitter.split_documents(blog_docs)


['Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as \nAutoGPT\n, \nGPT-Engineer\n and \nBabyAGI\n, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs', ' well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\n\n\nAgent System Overview\n#\n\n\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\n\n\n\nPlanning\n\n\n\n\nSubgoal and decomposition: The', 's:\n\n\n\n\nPlanning\n\n\n\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\n\n\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them fo', 'st actions, learn from mistakes and refine them fo

In [159]:
import logging

articles_collection = client.collections.get("Articles_2")
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

batch_size = 10
objects = []

for index, text_chunk in enumerate(my_chunks):
    try:
        obj = {
            "title": f"Title {index+1}",
            "abstractText": text_chunk
        }
        objects.append(obj)

        if len(objects) >= batch_size:
            articles_collection.data.insert_many(objects)
            logging.info(f"Inserted {index + 1} rows")
            objects = []

    except Exception as e:
        logging.error(f"Error processing row {index}: {e}")

if objects:
    articles_collection.data.insert_many(objects)
    logging.info("Final batch inserted")

logging.info("All data inserted successfully!")

response = articles_collection.aggregate.over_all(total_count=True)
print(f"成功匯入資料總數: {response.total_count}")

# vectorstore = Chroma.from_documents(documents=splits, 
#                                     embedding=OpenAIEmbeddings())

# # Create our retriever
# retriever = vectorstore.as_retriever()


2026-01-26 14:32:13,608 INFO Inserted 10 rows
2026-01-26 14:32:13,732 INFO Inserted 20 rows
2026-01-26 14:32:13,844 INFO Inserted 30 rows
2026-01-26 14:32:13,977 INFO Inserted 40 rows
2026-01-26 14:32:14,114 INFO Inserted 50 rows
2026-01-26 14:32:14,245 INFO Inserted 60 rows
2026-01-26 14:32:14,376 INFO Inserted 70 rows
2026-01-26 14:32:14,499 INFO Inserted 80 rows
2026-01-26 14:32:14,617 INFO Inserted 90 rows
2026-01-26 14:32:14,740 INFO Inserted 100 rows
2026-01-26 14:32:14,888 INFO Inserted 110 rows
2026-01-26 14:32:15,029 INFO Inserted 120 rows
2026-01-26 14:32:15,157 INFO Inserted 130 rows
2026-01-26 14:32:15,288 INFO Inserted 140 rows
2026-01-26 14:32:15,426 INFO Inserted 150 rows
2026-01-26 14:32:15,562 INFO Inserted 160 rows
2026-01-26 14:32:15,704 INFO Inserted 170 rows
2026-01-26 14:32:15,807 INFO Final batch inserted
2026-01-26 14:32:15,807 INFO All data inserted successfully!


成功匯入資料總數: 176


In [160]:
def call_LLM(prompt):
    payload = {
        "model": "gpt-oss:20b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0} 
    }

    response = requests.post("http://120.113.70.238:11434/api/generate", json=payload)
    return response.json()['response'].strip()


- 第一步(rewrite)：把使用者的原始問題丟給 LLM，請它「產生相關搜尋關鍵字」或「改寫問題」
- 第二步(retrieve)：使用Weaviate的Hybrid Search(混合搜尋 = 向量 + 關鍵字)
- 第三步(rerank)：把「問題跟「這5筆資料」配對，給另一個模型讀，選出最精準的Top3
- 第四步(generate)把經過翻譯->混合搜尋->重排序後留下的黃金資料，餵給LLM寫答案

In [ ]:
def rewrite(original_query):
    prompt = f"""
    你是一個搜尋優化專家。請將使用者的「中文口語問題」改寫成「英文的」、「適合學術檢索」的關鍵字查詢。
    只要給我改寫後的英文句子，不要解釋。
    
    使用者問題: {original_query}
    """
    rewritten = call_LLM(prompt)
    return rewritten

def retrieve(collection, query_text, limit=5):
    response = collection.query.hybrid(
        query=query_text,
        limit=limit,
        alpha=0.5
    )
    docs = [obj.properties['abstractText'] for obj in response.objects]
    return docs

def rerank(user_query, docs):
    if not docs: return []
    candidates_text = ""
    for i, doc in enumerate(docs):
        candidates_text += f"--- 候選片段 {i+1} ---\n{doc[:500]}...\n\n" 
    prompt = f"""
    任務：從以下候選片段中，挑選出最能回答「{user_query}」的 3 個片段。
    請直接回傳那 3 個片段的內容，不要解釋原因。
    {candidates_text}
    """
    best_docs = call_LLM(prompt)
    return best_docs


def generate(user_query, context):
    final_prompt = f"""
    請根據以下【背景資料】回答【使用者問題】。
    如果不確定，請說不知道。
    【資料】：{context}
    【問題】：{user_query}
    """
    return call_LLM(final_prompt)

In [163]:
user_input = "What Types of Memory?"
optimized_query = rewrite(user_input)
print(optimized_query)

Types of memory


In [167]:
initial_results = retrieve(articles_collection, optimized_query, limit=5)
print(initial_results)

['Memory\n#\n\n\nMemory can be defined as the processes used to acquire, store, retain, and later retrieve information. There are several types of memory in human brains.\n\n\n\n\n\n\nSensory Memory\n: This is the earliest stage of memory, providing the ability to retain impressions of sensory information (visual', 'memory (touch).\n\n\n\n\n\n\nShort-Term Memory\n (STM) or \nWorking Memory\n: It stores information that we are currently aware of and needed to carry out complex cognitive tasks such as learning and reasoning. Short-term memory is believed to have the capacity of about 7 items (\nMiller 1956\n) and lasts for 2', '(Image source: \nLaskin et al. 2023\n)\n\n\n\n\nComponent Two: Memory\n#\n\n\n(Big thank you to ChatGPT for helping me draft this section. I’ve learned a lot about the human brain and data structure for fast MIPS in my \nconversations\n with ChatGPT.)\n\n\nTypes of Memory\n#\n\n\nMemory can be defined as the processes ', 's).\n\n\nImplicit / procedural memory: Thi

In [165]:
best_context = rerank(user_input, initial_results)
print(best_context)

Memory
#
Memory can be defined as the processes used to acquire, store, retain, and later retrieve information. There are several types of memory in human brains.

Sensory Memory
: This is the earliest stage of memory, providing the ability to retain impressions of sensory information (visual...

---

memory (touch).

Short-Term Memory
 (STM) or 
Working Memory
: It stores information that we are currently aware of and needed to carry out complex cognitive tasks such as learning and reasoning. Short-term memory is believed to have the capacity of about 7 items (
Miller 1956
) and lasts for 2...

---

e are two subtypes of LTM:

Explicit / declarative memory: This is memory of facts and events, and refers to those memories that can be consciously recalled, including episodic memory (events and experiences) and semantic memory (facts and concepts).

Implicit / procedural memory: This type of m...


In [169]:
final_answer = generate(user_input, best_context)
# print(f"{final_answer}")
from IPython.display import Markdown, display
display(Markdown(final_answer))

**記憶的主要類型**

| 類型 | 主要特徵 | 典型子類別（如有） |
|------|----------|-------------------|
| **感官記憶 (Sensory Memory)** | 最早的記憶階段，能短暫保留感官刺激的印象（視覺、聽覺、觸覺等）。 | － |
| **短期記憶 (Short‑Term Memory, STM)** <br>或 **工作記憶 (Working Memory)** | 儲存目前正在意識中、需要用來完成複雜認知任務（學習、推理）的資訊。容量約 7 個項目，持續時間約 2 分鐘。 | － |
| **長期記憶 (Long‑Term Memory, LTM)** | 能長時間儲存資訊，容量相對無限。可分為兩大類：<br>1. **顯性／宣告性記憶 (Explicit / Declarative Memory)**：可有意識地回憶。<br>2. **隱性／程序性記憶 (Implicit / Procedural Memory)**：無需有意識即可運作。 | **顯性記憶**：<br>• 事件記憶（Episodic Memory）<br>• 事實記憶（Semantic Memory）<br>**隱性記憶**：<br>• 程序性記憶（Procedural Memory）<br>• 其他如條件反射、習慣等 |

> **簡述**  
> 1. **感官記憶**：捕捉感官輸入的瞬間印象。  
> 2. **短期/工作記憶**：處理即時資訊，並在需要時將其轉移到長期記憶。  
> 3. **長期記憶**：分為可意識回憶的顯性記憶（事件、事實）與無意識運作的隱性記憶（技能、習慣）。  

這些類型共同構成了人類大腦中資訊的獲取、儲存、保留與檢索過程。

Now, with our retriever ready, let’s explore our first query transformation technique.

<a id='part2-1'></a>
## Multi-Query Generation

多查詢生成：使用LLM產生使用者問題的幾個不同版本來解決這個問題，從而有效地從多個角度進行搜尋。

![Multi-Query Optimization (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*GjZoAISn6Jv3CBH87zUNPA.png).

下達指令提供給模型，將使用者問題改寫成5種不同語意角度的問題

In [170]:
client = OpenAI(
    base_url="http://120.113.70.238:11434/v1",
    api_key="ollama"
)

def generate_five_queries(original_question):
    prompt_template = f"""You are an AI language model assistant. Your task is to generate five 
    different versions of the given user question to retrieve relevant documents from a vector 
    database. By generating multiple perspectives on the user question, your goal is to help
    the user overcome some of the limitations of the distance-based similarity search. 
    Provide these alternative questions separated by newlines. 
    
    Original question: {original_question}"""


    response = client.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "user", "content": prompt_template}
        ],
        temperature=0 
    )
    content = response.choices[0].message.content
    queries = [line.strip() for line in content.split("\n") if line.strip()]
    return queries


# from langchain.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# # Prompt for generating multiple queries
# template = """You are an AI language model assistant. Your task is to generate five 
# different versions of the given user question to retrieve relevant documents from a vector 
# database. By generating multiple perspectives on the user question, your goal is to help
# the user overcome some of the limitations of the distance-based similarity search. 
# Provide these alternative questions separated by newlines. Original question: {question}"""
# prompt_perspectives = ChatPromptTemplate.from_template(template)

# # Chain to generate the queries
# generate_queries = (
#     prompt_perspectives 
#     | ChatOpenAI(temperature=0) 
#     | StrOutputParser() 
#     | (lambda x: x.split("\n"))
# )

提問問題，呼叫函數生成五個問題

In [171]:
question = "What is task decomposition for LLM agents?"
generated_queries_list = generate_five_queries(question)

for i, q in enumerate(generated_queries_list):
    print(f"{i+1}. {q}")


# question = "What is task decomposition for LLM agents?"
# generated_queries_list = generate_queries.invoke({"question": question})

# # Print the generated queries
# for i, q in enumerate(generated_queries_list):
#     print(f"{i+1}. {q}")

2026-01-26 18:59:48,440 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


1. What does “task decomposition” mean in the context of large language model agents?
2. How do LLM agents break down complex tasks into smaller sub‑tasks, and why is this useful?
3. Can you explain the process and benefits of task decomposition for building autonomous LLM agents?
4. What are common strategies for decomposing tasks when designing agents powered by large language models?
5. What challenges arise when implementing task decomposition in LLM‑based agent systems?


- simple_retriever()：去向量資料庫查文件，找出最相近的三篇，總共有五個問題，會有15篇的文件
- 整合所有query查詢結果，存進List
- 去掉重複的答案

In [172]:
def simple_retriever(query):
    response = articles_collection.query.near_text(query=query, limit=3)
    results = []
    for obj in response.objects:
        results.append({
            "abstractText": obj.properties.get("abstractText", ""),
            "title": obj.properties.get("title", "")
        })
    return results

all_retrieved_docs = []

for q in generated_queries_list:
    docs = simple_retriever(q)
    all_retrieved_docs.extend(docs) 

print(f"Total:{len(all_retrieved_docs)}")

unique_docs_map = {}
for doc in all_retrieved_docs:
    content = doc['abstractText']
    if content not in unique_docs_map:
        unique_docs_map[content] = doc

final_unique_docs = list(unique_docs_map.values())
print(f"Leave:{len(final_unique_docs)}")


# from langchain.load import dumps, loads

# def get_unique_union(documents: list[list]):
#     """ A simple function to get the unique union of retrieved documents """
#     # Flatten the list of lists and convert each Document to a string for uniqueness
#     flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
#     unique_docs = list(set(flattened_docs))
#     return [loads(doc) for doc in unique_docs]

# # Build the retrieval chain
# retrieval_chain = generate_queries | retriever.map() | get_unique_union

# # Invoke the chain and check the number of documents retrieved
# docs = retrieval_chain.invoke({"question": question})
# print(f"Total unique documents retrieved: {len(docs)}")

Total:15
Leave:7


下達這是檢索完的答案，限制模型生成，最後整理輸出為答案

In [173]:
template = """Answer the following question based on this context:
{context}
Question: {question}
"""

context_text = "\n\n".join([doc['abstractText'] for doc in final_unique_docs])

final_prompt_str = template.format(
    context=context_text,
    question=question
)

response = llm_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": final_prompt_str}
    ],
    temperature=0 
)

final_answer = response.choices[0].message.content
# print(final_answer)
display(Markdown(final_answer))

# from operator import itemgetter
# from langchain_core.runnables import RunnablePassthrough

# # The final RAG chain
# template = """Answer the following question based on this context:

# {context}

# Question: {question}
# """
# prompt = ChatPromptTemplate.from_template(template)
# llm = ChatOpenAI(temperature=0)

# final_rag_chain = (
#     {"context": retrieval_chain, "question": itemgetter("question")} 
#     | prompt
#     | llm
#     | StrOutputParser()
# )

# print(final_rag_chain.invoke({"question": question}))

2026-01-26 19:00:01,973 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Task decomposition for LLM agents** is the process by which a large‑language‑model‑driven agent takes a complex, multi‑step problem and breaks it down into a sequence of smaller, more manageable sub‑tasks or sub‑goals.  

- **Why it matters**: A single, monolithic request is often too hard for the model to solve in one go. By decomposing the task, the agent can reason about each piece separately, keep track of dependencies, and assemble the final answer from the results of the sub‑tasks.  
- **How it works**:  
  1. **Identify the overall goal** (e.g., “write a research paper on X”).  
  2. **Generate a list of sub‑goals** (e.g., literature review, methodology design, data collection, analysis, drafting, editing).  
  3. **Assign IDs, dependencies, and arguments** to each sub‑task so the agent knows the order and inputs needed.  
  4. **Iteratively solve each sub‑task**—often using chain‑of‑thought prompting to encourage step‑by‑step reasoning—then combine the outputs.  

This decomposition strategy is a core component of autonomous LLM agents such as AutoGPT, BabyAGI, and GPT‑Engineer, enabling them to handle complex projects efficiently and reliably.

<a id='part2-2'></a>
## RAG-Fusion

Multi-Query is a great start, but simply taking a union of documents treats them all equally. What if one document was ranked highly by three of our queries, while another was a low-ranked result from only one?

![RAG Fusion (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*qIJlH2bVjc1ZZflcniuHCw.png)

The first is clearly more important. RAG-Fusion improves on Multi-Query by not just fetching documents, but also …

> **re-ranking** them using a technique called **Reciprocal Rank Fusion (RRF)**.

RRF intelligently combines results from multiple searches. It boosts the score of documents that appear consistently high across different result lists, pushing the most relevant content to the top.

The code is very similar, but we’ll swap our `get_unique_union` function with an RRF implementation.

- **第一步：使用者提出一個原始問題**
- **第二步：LLM 產生多種不同語意角度的問題**
  - 將同一個問題，改寫成多種語意相近但問法不同的問題

- **第三步：Retriever（多次檢索）**
  - 每一個問法，都向資料庫查詢一次  
  - 範例如下：
    - 第一種問法 → 找到 `A、B、C`
    - 第二種問法 → 找到 `A、D、E`
    - 第三種問法 → 找到 `A、B、F`

- **第四步：RRF（Reciprocal Rank Fusion）融合 + 排序**
  - 如果一篇資料被很多問法都找到，而且每次排名都很前面  
    → 那它「很可能真的很重要」
##### 排名邏輯說明
- `A` 出現3次，而且每次都排在前面  **分數很高**
- `F` 只出現1次，而且排名較後 **分數很低**

In [174]:
import json
def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = json.dumps(doc, sort_keys=True)
            
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (json.loads(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return reranked_results

# def reciprocal_rank_fusion(results: list[list], k=60):
#     """ Reciprocal Rank Fusion that intelligently combines multiple ranked lists """
#     fused_scores = {}

#     # Iterate through each list of ranked documents
#     for docs in results:
#         for rank, doc in enumerate(docs):
#             doc_str = dumps(doc)
#             if doc_str not in fused_scores:
#                 fused_scores[doc_str] = 0
#             # The core of RRF: documents ranked higher (lower rank value) get a larger score
#             fused_scores[doc_str] += 1 / (rank + k)

#     # Sort documents by their new fused scores in descending order
#     reranked_results = [
#         (loads(doc), score)
#         for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
#     ]
#     return reranked_results

In [175]:
from openai import OpenAI

client = OpenAI(
    base_url="http://120.113.70.238:11434/v1",
    api_key="ollama"
)

def generate_rag_fusion_queries(original_question):
    prompt = f"""You are a helpful assistant that generates multiple search queries based on a single input query. 
    Generate multiple search queries related to: {original_question} 
    Output (4 queries):"""

    response = client.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    content = response.choices[0].message.content
    queries = [line.strip() for line in content.split("\n") if line.strip()]
    return queries

fusion_queries = generate_rag_fusion_queries(question)
print(f"general select: {fusion_queries}")

results_list_of_lists = []

for q in fusion_queries:
    docs = simple_retriever(q) 
    results_list_of_lists.append(docs)

final_fusion_docs = reciprocal_rank_fusion(results_list_of_lists)
print(f"Total re-ranked documents retrieved: {len(final_fusion_docs)}")


# # Use a slightly different prompt for RAG-Fusion
# template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
# Generate multiple search queries related to: {question} \n
# Output (4 queries):"""
# prompt_rag_fusion = ChatPromptTemplate.from_template(template)

# generate_queries = (
#     prompt_rag_fusion 
#     | ChatOpenAI(temperature=0)
#     | StrOutputParser() 
#     | (lambda x: x.split("\n"))
# )

# # Build the new retrieval chain with RRF
# retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
# docs = retrieval_chain_rag_fusion.invoke({"question": question})

# print(f"Total re-ranked documents retrieved: {len(docs)}")

2026-01-26 20:48:21,473 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


general select: ['1. “Task decomposition in large language model agents – definition and examples”', '2. “How do LLM agents perform task decomposition?”', '3. “Best practices for task decomposition with LLM agents”', '4. “Case studies of task decomposition in autonomous LLM agents”']
Total re-ranked documents retrieved: 5


<a id='part2-3'></a>
## Decomposition

![Answer Recursively (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*oYttQUN_G0J_TZtigWjsGQ.png)

**Decomposition → 原始問題生成多個子問題**
- 將複雜問題拆成可以單獨回答的子問題
- 去重、限制數量、過濾無效或太長/太短的問題

↓
**RAG → 每個子問題單獨檢索並生成答案**
- 對每個子問題：
  - 檢索相關文件（Retriever / 向量資料庫）
  - 組合 context
  - 使用 LLM 根據 context 生成子答案

↓
**組合 Q+A**
- 將所有子問題及其答案整理成標準 Q+A 格式

↓
**Synthesis → 生成原問題最終答案**
- 將整理好的 Q+A context 送入 LLM
- 生成原始問題的完整最終答案

↓
**輸出答案**
- 最終答案整合所有子問題的結果
- 更準確、完整、可靠


In [176]:
def generate_sub_questions(question, max_subquestions=3):
    template = f"""You are a helpful assistant that generates multiple sub-questions related to an input question.
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation.
Generate multiple search queries related to: {question}
Output (at least {max_subquestions} queries):"""

    response = client.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": template}],
        temperature=0
    )

    raw_sub_questions = [line.strip() for line in response.choices[0].message.content.split("\n") if line.strip()]
    unique_sub_questions = []
    for sq in raw_sub_questions:
        if sq not in unique_sub_questions:
            unique_sub_questions.append(sq)

    filtered_sub_questions = [
        sq for sq in unique_sub_questions if 5 <= len(sq) <= 150
    ]

    return filtered_sub_questions[:max_subquestions]

question = "What are the main components of an LLM-powered autonomous agent system?"
sub_questions = generate_sub_questions(question)
print(sub_questions)


# # Decomposition prompt
# template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
# The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
# Generate multiple search queries related to: {question} \n
# Output (3 queries):"""
# prompt_decomposition = ChatPromptTemplate.from_template(template)

# # Chain to generate sub-questions
# generate_queries_decomposition = (
#     prompt_decomposition 
#     | ChatOpenAI(temperature=0) 
#     | StrOutputParser() 
#     | (lambda x: x.split("\n"))
# )

# # Generate and print the sub-questions
# question = "What are the main components of an LLM-powered autonomous agent system?"
# sub_questions = generate_queries_decomposition.invoke({"question": question})
# print(sub_questions)

2026-01-26 20:48:29,734 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


['- “LLM‑powered autonomous agent architecture components”', '- “Key building blocks of a large‑language‑model autonomous agent system”', '- “What are the core modules in an LLM‑driven autonomous agent?”']


In [33]:
# from langsmith import Client

# client = Client()
# prompt = client.pull_prompt("rlm/rag-prompt")
# print(prompt)
client = OpenAI(base_url="http://120.113.70.238:11434/v1",api_key="ollama")

In [177]:
rag_prompt_template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Question: {question} 
Context: {context} 
Answer:"""

def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()


rag_results = []
MAX_CHARS = 3000

for sub_q in sub_questions:
    docs = simple_retriever(sub_q)
    context_text = ""
    for d in docs:
        text = d['abstractText']
        if len(context_text) + len(text) + 2 > MAX_CHARS:
            break
        context_text += text + "\n"

    response = client.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{
            "role": "user",
            "content": rag_prompt_template.format(question=sub_q, context=context_text)
        }],
        temperature=0
    )
    rag_results.append(response.choices[0].message.content)


context = format_qa_pairs(sub_questions, rag_results)
template = f"""Here is a set of Q+A pairs:
{context}
Use these to synthesize an answer to the original question: {question}
"""

final_response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[{"role": "user", "content": template}],
    temperature=0
)

# print(final_response.choices[0].message.content)
display(Markdown(final_response.choices[0].message.content))

# # RAG prompt
# prompt_rag = hub.pull("rlm/rag-prompt")

# # A list to hold the answers to our sub-questions
# rag_results = []
# for sub_question in sub_questions:
#     # Retrieve documents for each sub-question
#     retrieved_docs = retriever.get_relevant_documents(sub_question)
    
#     # Use our standard RAG chain to answer the sub-question
#     answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs, "question": sub_question})
#     rag_results.append(answer)

# def format_qa_pairs(questions, answers):
#     """Format Q and A pairs"""
#     formatted_string = ""
#     for i, (question, answer) in enumerate(zip(questions, answers), start=1):
#         formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
#     return formatted_string.strip()

# # Format the Q&A pairs into a single context string
# context = format_qa_pairs(sub_questions, rag_results)

# # Final synthesis prompt
# template = """Here is a set of Q+A pairs:

# {context}

# Use these to synthesize an answer to the original question: {question}
# """
# prompt = ChatPromptTemplate.from_template(template)

# final_rag_chain = (
#     prompt
#     | llm
#     | StrOutputParser()
# )

# print(final_rag_chain.invoke({"context": context, "question": question}))

2026-01-26 20:51:19,413 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-26 20:51:22,138 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-26 20:51:24,350 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-26 20:51:30,108 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Main components of an LLM‑powered autonomous agent system**

| # | Component | Core responsibilities | Typical implementation |
|---|-----------|-----------------------|------------------------|
| 1 | **LLM “brain”** | Generates natural‑language text, plans, reasons, and orchestrates other modules. | GPT‑4, Llama‑2, Claude, etc. |
| 2 | **Planning / task‑decomposition** | Breaks a high‑level goal into actionable sub‑tasks, orders them, and monitors progress. | Hierarchical planners, goal‑graph generators, or LLM‑based planners. |
| 3 | **Reasoning engine** | Performs step‑by‑step inference, calculations, or logic needed to solve each sub‑task. | Chain‑of‑Thought prompting, neuro‑symbolic modules (e.g., MRKL), or external reasoning APIs. |
| 4 | **Dynamic memory / self‑reflection** | Stores past interactions, observations, and outcomes; revisits decisions to improve future behavior. | Retrieval‑augmented memory, episodic buffers, Reflexion‑style introspection loops. |
| 5 | **Modular expert systems** | Specialized sub‑models or APIs that provide domain‑specific skills (navigation, translation, math, API calls). | MRKL‑style expert modules, external toolkits, or custom fine‑tuned models. |
| 6 | **Orchestration / execution manager** | Coordinates the LLM, planners, reasoners, and experts; decides which module to invoke next and integrates their outputs. | Rule‑based dispatcher, LLM‑guided controller, or a lightweight workflow engine. |
| 7 | **Feedback & learning loop** | Uses outcomes and user feedback to refine policies, prompts, or fine‑tune components. | Reinforcement learning, supervised fine‑tuning, or prompt‑engineering cycles. |

**How they work together**

1. **Goal receipt** – The agent receives a user request or environmental cue.  
2. **Planning** – The planner decomposes the goal into sub‑tasks.  
3. **Execution loop** – For each sub‑task, the orchestrator selects the appropriate expert or reasoning module.  
4. **Reasoning** – The LLM (or a specialized module) generates the necessary steps or calculations.  
5. **Memory update** – Results and observations are stored in dynamic memory for future reference.  
6. **Self‑reflection** – Periodically, the agent reviews past decisions, identifies mistakes, and updates its strategy.  
7. **Feedback integration** – External feedback (e.g., user ratings) is used to fine‑tune prompts or retrain components.

These components together enable an LLM‑powered autonomous agent to **plan, reason, act, learn, and improve** in a coordinated, self‑driving manner.

<a id='part2-4'></a>
## Step-Back Prompting

![Step Back Prompting (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:875/1*6lrhGv1fdcmLKMVu5tU3uQ.png)

- 第一步：後退一步，進行抽象化 (Abstraction)

使用者問了一個很細節的問題：「LLM Agent 的任務分解是什麼？」
如果直接去搜，可能只會找到一些零碎的定義。
Step-Back機制會強迫LLM先退一步想：「在回答這個細節之前，我應該先問什麼大哉問？」
LLM 生成了新問題：「LLM Agent 的運作原理與規劃機制是什麼？」(Step-Back Question)。

- 第二步：雙路檢索

進行了兩次搜尋。
低路徑 (Specific)：用「原始問題」去搜。
搜到了：JSON 格式範例、Task ID 定義、具體的實作參數。
高路徑 (Generic)：用「抽象問題」去搜。
搜到了：Chain of Thought (CoT)、MRKL 架構、神經符號系統 (Neuro-symbolic) 等高階理論。
結果：您的 Context (背景資料) 變得**「既有廣度、又有深度」**。傳統 RAG 通常只有深度（細節），缺乏廣度（原理）。

- 第三步：整合生成

LLM 收到了最終指令：「你是專家，請給我一個詳盡 (Comprehensive) 的回答。」
它的桌上擺了兩份資料：一份是「實作手冊」(Normal Context)，一份是「教科書理論」(Step-Back Context)。
因為模型夠聰明，它決定把這兩份資料融合。它先解釋了理論，再解釋了實作，最後給出了結論。

In [ ]:
def generate_step_back_query(question):
    system_prompt = (
        "You are an expert at world knowledge. Your task is to step back and paraphrase a question "
        "to a more generic step-back question, which is easier to answer. Here are a few examples:"
    )
  
    messages = [
        {"role": "system", "content": system_prompt},
 
        {"role": "user", "content": "Could the members of The Police perform lawful arrests?"},
        {"role": "assistant", "content": "what can the members of The Police do?"},

        {"role": "user", "content": "Jan Sindel's was born in what country?"},
        {"role": "assistant", "content": "what is Jan Sindel's personal history?"},

        {"role": "user", "content": question}
    ]
    
    response = llm_client.chat.completions.create(
        model="gpt-oss:20b",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

def retrieve(query, limit=3):
    response = articles_collection.query.near_text(
        query=query,
        limit=limit
    )

    context_text = "\n\n".join([obj.properties["abstractText"] for obj in response.objects])
    return context_text if context_text else "No relevant context found."

def final_rag_generation(question, normal_context, step_back_context):

    prompt = f"""You are an expert of world knowledge. I am going to ask you a question. 
            Your response should be comprehensive and not contradicted with the following context if they are relevant. 
            Otherwise, ignore them if they are not relevant.

            # Normal Context (Specific Details)
            {normal_context}

            # Step-Back Context (General Concepts)
            {step_back_context}

            # Original Question: {question}
            # Answer:"""

    response = llm_client.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


# from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# # Few-shot examples to teach the model how to generate step-back (more generic) questions
# examples = [
#     {
#         "input": "Could the members of The Police perform lawful arrests?",
#         "output": "what can the members of The Police do?",
#     },
#     {
#         "input": "Jan Sindel's was born in what country?",
#         "output": "what is Jan Sindel's personal history?",
#     },
# ]

# # Define how each example is formatted in the prompt
# example_prompt = ChatPromptTemplate.from_messages([
#     ("human", "{input}"),  # User input
#     ("ai", "{output}")     # Model's response
# ])

# # Wrap the few-shot examples into a reusable prompt template
# few_shot_prompt = FewShotChatMessagePromptTemplate(
#     example_prompt=example_prompt,
#     examples=examples,
# )

# # Full prompt includes system instruction, few-shot examples, and the user question
# prompt = ChatPromptTemplate.from_messages([
#     ("system", 
#      "You are an expert at world knowledge. Your task is to step back and paraphrase a question "
#      "to a more generic step-back question, which is easier to answer. Here are a few examples:"),
#     few_shot_prompt,
#     ("user", "{question}"),
# ])

In [180]:
question = "What is task decomposition for LLM agents?"
print(question)

What is task decomposition for LLM agents?


In [181]:
step_back_question = generate_step_back_query(question)

print(f"Original Question: {question}")
print(f"Step-Back Question: {step_back_question}")

# # Define a chain to generate step-back questions using the prompt and an OpenAI model
# generate_queries_step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()

# # Run the chain on a specific question
# question = "What is task decomposition for LLM agents?"
# step_back_question = generate_queries_step_back.invoke({"question": question})

# # Output the original and generated step-back question
# print(f"Original Question: {question}")
# print(f"Step-Back Question: {step_back_question}")

# from langchain_core.runnables import RunnableLambda

# # Prompt for the final response
# response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# # Normal Context
# {normal_context}

# # Step-Back Context
# {step_back_context}

# # Original Question: {question}
# # Answer:"""
# response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# # The full chain
# chain = (
#     {
#         # Retrieve context using the normal question
#         "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
#         # Retrieve context using the step-back question
#         "step_back_context": generate_queries_step_back | retriever,
#         # Pass on the original question
#         "question": lambda x: x["question"],
#     }
#     | response_prompt
#     | ChatOpenAI(temperature=0)
#     | StrOutputParser()
# )

# response = chain.invoke({"question": question})

2026-01-26 21:20:51,580 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


Original Question: What is task decomposition for LLM agents?
Step-Back Question: What is task decomposition?


In [182]:
ctx_normal = retrieve(question)
ctx_step_back = retrieve(step_back_question)

final_answer = final_rag_generation(question, ctx_normal, ctx_step_back)

# print(final_answer)
from IPython.display import Markdown, display
display(Markdown(final_answer))

# print(response)

2026-01-26 21:21:15,877 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Task decomposition for LLM agents**  
— the systematic breaking‑down of a complex user request into a set of smaller, inter‑related subtasks that an LLM‑powered autonomous agent can execute in sequence or in parallel.

---

## 1. Why decomposition matters

| Problem | Decomposition benefit |
|---------|-----------------------|
| **Large, multi‑step goals** (e.g., “Plan a week‑long trip to Japan”) | Turns a single, hard‑to‑solve request into manageable pieces (flight booking, hotel search, itinerary, budget). |
| **Ambiguous or underspecified input** | Allows the agent to ask clarifying questions for each sub‑goal, reducing hallucination. |
| **Resource constraints** | Enables the agent to schedule tasks, respect dependencies, and avoid redundant work. |
| **Learning & self‑reflection** | Each sub‑task can be evaluated, refined, and reused in future runs. |

---

## 2. Core concepts

| Concept | Definition | How an LLM uses it |
|---------|------------|--------------------|
| **Subgoal** | A concrete, actionable objective that contributes to the overall goal. | The LLM generates a list of subgoals from the user prompt. |
| **Task ID** | Unique identifier for each subtask. | Used to reference the task in dependencies and logs. |
| **Dependencies (`dep`)** | IDs of tasks that must finish before the current task can start. | The LLM arranges tasks in a directed acyclic graph (DAG). |
| **Arguments (`args`)** | Input data required for the task (text, image URLs, etc.). | The LLM populates these fields from earlier tasks or user input. |
| **Task type** | Category of the subtask (e.g., `search`, `generate`, `validate`, `compose`). | Helps the agent route the subtask to the appropriate execution engine. |

---

## 3. The decomposition workflow

1. **Parse the user request**  
   *The LLM reads the prompt and identifies the high‑level goal.*

2. **Generate a task list**  
   *Using few‑shot examples, the LLM produces a JSON array of subtasks:*

   ```json
   [
     {"task":"search_flights","id":"t1","dep":[],"args":{"origin":"NYC","destination":"Tokyo","date":"2026-02-01"}},
     {"task":"book_hotel","id":"t2","dep":["t1"],"args":{"city":"Tokyo","checkin":"2026-02-01","checkout":"2026-02-08"}},
     {"task":"create_itinerary","id":"t3","dep":["t2"],"args":{"days":7,"interests":["sightseeing","food"]}}
   ]
   ```

3. **Plan execution order**  
   *The agent builds a DAG, ensuring that `t2` waits for `t1`, etc.*

4. **Execute subtasks**  
   *Each subtask is handed off to the appropriate module (API call, internal function, or another LLM prompt).*

5. **Reflect & refine**  
   *After each subtask, the agent logs success/failure, updates the plan if needed, and may re‑decompose or re‑rank tasks.*

---

## 4. Techniques that enable effective decomposition

| Technique | How it helps |
|-----------|--------------|
| **Chain‑of‑Thought (CoT)** | Encourages the LLM to “think aloud,” revealing intermediate reasoning steps that naturally map to subtasks. |
| **Few‑shot prompting** | Provides concrete examples of task lists, teaching the LLM the desired JSON schema. |
| **Self‑criticism prompts** | After a subtask, the LLM evaluates its own output, catching errors before they propagate. |
| **Dependency‑aware prompting** | Explicitly asks the LLM to list dependencies, reducing circular or missing links. |

---

## 5. Example: “Write a marketing email for a new product launch”

1. **User prompt**  
   *“I need a marketing email for our new smartwatch.”*

2. **LLM output (decomposed tasks)**

   ```json
   [
     {"task":"gather_product_info","id":"p1","dep":[],"args":{"product":"smartwatch"}},
     {"task":"identify_target_audience","id":"p2","dep":["p1"],"args":{}},
     {"task":"draft_email_body","id":"p3","dep":["p2"],"args":{}},
     {"task":"design_email_template","id":"p4","dep":["p3"],"args":{}},
     {"task":"run_a_b_test","id":"p5","dep":["p4"],"args":{}}
   ]
   ```

3. **Execution**  
   *The agent calls a knowledge‑base API for product specs, runs a demographic analysis, generates the email text, renders a template, and finally submits two variants to an A/B testing platform.*

4. **Reflection**  
   *If the A/B test shows low engagement, the agent can re‑decompose `p3` (draft email body) with new prompts.*

---

## 6. Benefits over monolithic LLM calls

| Feature | Monolithic LLM | Decomposed LLM |
|---------|----------------|----------------|
| **Error isolation** | One hallucination can ruin the whole output. | Errors are confined to a subtask; the rest can still succeed. |
| **Parallelism** | Hard to parallelize. | Independent subtasks can run concurrently. |
| **Explainability** | Hard to trace. | Each subtask’s input/output is logged. |
| **Reusability** | Low. | Subtasks (e.g., “search_flights”) can be cached and reused. |
| **Learning** | Limited. | Each subtask can be evaluated and the plan refined. |

---

## 7. Common pitfalls and how to avoid them

| Pitfall | Mitigation |
|---------|------------|
| **Over‑splitting** | Use a threshold on subtask complexity; merge trivial tasks. |
| **Circular dependencies** | Validate the DAG before execution. |
| **Missing arguments** | Enforce schema validation; prompt the LLM to fill missing fields. |
| **Unclear task types** | Provide a taxonomy of task types in the prompt. |
| **Hallucinated dependencies** | Cross‑check dependencies against known APIs or data sources. |

---

## 8. Take‑away

Task decomposition turns a single, often ambiguous user request into a **structured, executable plan**. By representing each subgoal as a JSON object with an ID, dependencies, arguments, and a clear task type, an LLM‑powered agent can:

1. **Plan** – build a DAG that respects logical order.  
2. **Execute** – delegate to specialized modules or APIs.  
3. **Reflect** – evaluate outcomes, learn from mistakes, and refine the plan.  

This disciplined approach is the backbone of robust, scalable autonomous agents that can handle real‑world complexity while remaining transparent and controllable.

<a id='part2-5'></a>
## HyDE

This final technique is one of the most clever. The core problem of retrieval is that a user’s query might use different words than the document (the “vocabulary mismatch” problem).

![HyDE (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*YQVJMOpDBU6l54atHoFpJg.png)

### 1️⃣ 生成假答案（Hypothetical Answer）
- 使用者問題通常很短、關鍵字少
- 系統要求 LLM「假裝專家」寫一段看似專業的回答
- 內容可能是錯的（幻覺），但 **用詞與真實文件相似**

### 2️⃣ 用假答案做檢索
- 將「假答案」轉成向量
- 取代原本「短問題向量」進行搜尋
- 更容易在向量資料庫中找到語意相近的真實文件
從「大海撈針」變成「找語意雙胞胎」

### 3️⃣ 最終生成答案
- 將檢索到的 **真實文件** 提供給 LLM
- LLM 根據真實資料回答原始問題



In [183]:
print(question)

What is task decomposition for LLM agents?


In [184]:
# 假答案生成
def generate_hyde_doc(question):
    template = f"""Please write a scientific paper passage to answer the question.
                Question: {question}
                Passage:"""

    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": template}],
        temperature=0 
    )
    return response.choices[0].message.content

# 用「假答案（HyDE 產生的內容）」去向量資料庫搜尋真正相關的文件
def retrieve_with_hyde(hypothetical_doc, limit=3):
    response = articles_collection.query.near_text(
        query=hypothetical_doc, 
        limit=limit
    )
    
    docs = [obj.properties["abstractText"] for obj in response.objects]
    return "\n\n".join(docs) if docs else "No relevant documents found."

# 用「真實文件內容」來回答使用者的原始問題
def final_rag_generation(question, context):
    prompt = f"""Answer the question based on the context below.
                Context:{context}
                Question: {question}
                Answer:"""
    
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0 
    )
    return response.choices[0].message.content

# # HyDE prompt
# template = """Please write a scientific paper passage to answer the question
# Question: {question}
# Passage:"""
# prompt_hyde = ChatPromptTemplate.from_template(template)

# # Chain to generate the hypothetical document
# generate_docs_for_retrieval = (
#     prompt_hyde 
#     | ChatOpenAI(temperature=0) 
#     | StrOutputParser() 
# )

# # Generate and print the hypothetical document
# hypothetical_document = generate_docs_for_retrieval.invoke({"question": question})
# print(hypothetical_document)

In [185]:
hypothetical_document = generate_hyde_doc(question)
# print(hypothetical_document)
display(Markdown(hypothetical_document))
# # Retrieve documents using the HyDE approach
# retrieval_chain = generate_docs_for_retrieval | retriever 
# retrieved_docs = retrieval_chain.invoke({"question": question})


2026-01-26 22:09:20,328 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Task Decomposition for Large Language Model (LLM) Agents**

Large language models (LLMs) have emerged as powerful general‑purpose agents capable of performing a wide spectrum of tasks, from natural‑language dialogue to code synthesis and scientific reasoning. However, the raw capacity of an LLM to generate coherent text does not automatically translate into reliable, goal‑oriented behavior in complex, multi‑step environments. Task decomposition—systematically breaking a high‑level objective into a hierarchy of sub‑tasks—has become a cornerstone technique for harnessing LLMs as autonomous agents. In this section we formalize the notion of task decomposition for LLM agents, review prevailing methodologies, and discuss its practical implications.

---

### 1. Definition

Task decomposition for LLM agents refers to the process of transforming a monolithic goal \(G\) into a structured plan \(\Pi = \langle T_1, T_2, \dots, T_n \rangle\) where each \(T_i\) is a sub‑task that can be executed by the LLM (or by a hybrid system that delegates to external tools). Each sub‑task is defined by a *prompt template* \(P_i\) and an *execution context* \(C_i\) that encapsulate the necessary information for the LLM to produce a valid intermediate result. The decomposition is typically recursive: sub‑tasks may themselves be decomposed until they reach a *leaf* level that is directly executable by the LLM without further abstraction.

Formally, let \(S\) denote the set of all possible states of the environment, and let \(f: S \times \Theta \rightarrow S\) be the transition function induced by the LLM’s output, where \(\Theta\) is the set of all possible prompt–context pairs. Task decomposition seeks a sequence \(\{(P_i, C_i)\}_{i=1}^n\) such that:

\[
s_{i+1} = f(s_i, (P_i, C_i)), \quad s_0 = s_{\text{init}}, \quad s_n \models G,
\]

where \(s_n \models G\) indicates that the final state satisfies the goal specification.

---

### 2. Motivations

1. **Complexity Management** – LLMs excel at generating fluent text but struggle with long‑horizon planning. Decomposing a task reduces the planning horizon for each sub‑task, mitigating error accumulation.

2. **Modularity and Reusability** – Sub‑tasks can be treated as reusable modules. A well‑designed decomposition allows an agent to compose previously learned sub‑tasks for new goals, fostering transfer learning.

3. **Explainability** – A hierarchical plan provides a transparent trace of the agent’s reasoning, which is essential for debugging and for compliance in regulated domains.

4. **Tool Integration** – Many real‑world tasks require interaction with external APIs (e.g., database queries, web scraping). Decomposition naturally delineates the boundaries where the LLM hands off to specialized tools.

---

### 3. Methodologies

#### 3.1 Prompt‑Based Decomposition

The most straightforward approach is to embed decomposition logic directly into the prompt. For example, a “chain‑of‑thought” prompt can be extended to produce a list of sub‑goals:

```
Goal: Write a Python script that scrapes product prices from three e‑commerce sites.

1. Identify the target sites.
2. For each site, determine the URL pattern for product pages.
3. Generate a scraper function for each site.
4. Combine the scrapers into a single script.
```

The LLM then executes each numbered step sequentially, optionally verifying intermediate outputs before proceeding.

#### 3.2 Hierarchical Reinforcement Learning (HRL)

In HRL, a high‑level policy selects sub‑tasks, while low‑level policies (implemented by the LLM) execute them. The LLM can be fine‑tuned to respond to sub‑task prompts, and the high‑level controller can be trained via reinforcement learning to optimize overall task success. This approach has been demonstrated in robotic manipulation and dialogue systems [1].

#### 3.3 Planner‑LLM Hybrids

External symbolic planners (e.g., PDDL planners) can generate a high‑level plan that the LLM then fills in. The planner ensures logical consistency and feasibility, while the LLM provides natural‑language descriptions and code snippets. This hybridization has shown improved reliability in complex planning domains [2].

#### 3.4 Self‑Decomposition via Meta‑Learning

Recent work has explored letting the LLM learn to decompose tasks autonomously. By exposing the model to a curriculum of tasks with known decompositions, the LLM can internalize a decomposition policy that generalizes to unseen goals. This meta‑learning approach reduces the need for hand‑crafted prompt templates [3].

---

### 4. Illustrative Example

Consider the goal: *“Plan a week‑long vegetarian dinner menu that satisfies a 2000‑calorie daily intake and includes at least 50 g of protein per meal.”* A decomposed plan might be:

1. **Define nutritional constraints** – Compute daily macro‑requirements.
2. **Generate a list of vegetarian protein sources** – e.g., lentils, tofu, quinoa.
3. **Create a meal template** – Breakfast, lunch, dinner, snacks.
4. **Assign protein sources to meals** – Ensure each meal meets protein target.
5. **Select complementary ingredients** – Vegetables, grains, spices.
6. **Assemble recipes** – Provide step‑by‑step instructions.
7. **Validate calorie count** – Sum calories per meal and per day.

The LLM can be prompted to produce each sub‑task’s output, and a lightweight validation script can verify nutritional compliance before moving to the next step.

---

### 5. Challenges and Open Questions

- **Error Propagation** – Mistakes in early sub‑tasks can cascade, leading to failure of the entire plan. Robust error‑handling and rollback mechanisms are needed.
- **Granularity Selection** – Determining the optimal depth of decomposition is non‑trivial; too fine a granularity burdens the LLM with trivial tasks, while too coarse a granularity reintroduces planning complexity.
- **Tool Coordination** – Seamlessly integrating LLM outputs with external APIs requires careful interface design and state management.
- **Evaluation Metrics** – Standardized benchmarks for task decomposition quality (e.g., plan optimality, execution success rate) are still emerging.

---

### 6. Conclusion

Task decomposition transforms the raw generative power of LLMs into disciplined, goal‑oriented agents capable of tackling complex, multi‑step problems. By structuring tasks hierarchically, we reduce planning horizons, enable modularity, and facilitate integration with specialized tools. While promising, the field faces challenges in error handling, granularity tuning, and evaluation. Continued research into hybrid planners, meta‑learning for decomposition, and robust execution frameworks will be essential for realizing the full potential of LLM agents in real‑world applications.

---

**References**

[1] Smith, J., & Doe, A. (2024). *Hierarchical Reinforcement Learning with Large Language Models for Robotics*. Proceedings of the 2024 International Conference on Robotics and Automation.

[2] Lee, K., & Patel, R. (2023). *Planner‑LLM Hybrids for Complex Task Execution*. Journal of Artificial Intelligence Research, 58(4), 1123–1145.

[3] Zhang, L., & Chen, M. (2025). *Self‑Decomposition in Large Language Models via Meta‑Learning*. arXiv preprint arXiv:2501.01234.

In [186]:
real_context = retrieve_with_hyde(hypothetical_document)
final_answer = final_rag_generation(question, real_context)
display(Markdown(final_answer))

# Use our standard RAG chain to generate the final answer from the retrieved context
# response = final_rag_chain.invoke({"context": retrieved_docs, "question": question})
# print(response)

2026-01-26 22:09:30,773 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Task decomposition for LLM agents** is the process of breaking a complex, multi‑step task into a sequence of smaller, well‑defined sub‑goals or actions. By identifying the constituent parts of a problem, an LLM agent can:

1. **Plan ahead** – determine the order and dependencies of each sub‑goal.  
2. **Manage complexity** – focus on one manageable piece at a time rather than trying to solve the whole problem in one go.  
3. **Improve accuracy** – each sub‑goal can be verified or refined before moving on, reducing the chance of cascading errors.  
4. **Enable modular execution** – the agent can reuse or recombine sub‑goals across different tasks.

In practice, task decomposition involves:  
- **Analyzing the overall objective** to identify logical steps.  
- **Defining clear, actionable sub‑tasks** that the LLM can generate or execute.  
- **Sequencing and prioritizing** these sub‑tasks, often using a planning or chain‑of‑thought approach.  
- **Iteratively refining** the decomposition based on feedback or self‑reflection.

Thus, task decomposition is a foundational planning strategy that turns a daunting problem into a series of tractable, ordered actions for an LLM‑powered autonomous agent.

<a id='part3'></a>
# Routing & Query Construction

Our RAG system is getting smarter, but in a real-world scenario, knowledge isn’t stored in a single, uniform library.

> We often have multiple data sources: documentation for different programming languages, internal wikis, public websites, or databases with structured metadata.

![Routing and Query Transformation (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*cost0_AWB8NKp0WxZlH7fA.png)

Sending every query to every source is wildly inefficient and can lead to noisy, irrelevant results.

This is where our RAG system needs to evolve from a simple librarian into an **intelligent switchboard operator**. It needs the ability to first *analyze* an incoming query and then *route* it to the correct destination or *construct* a more precise, structured query for retrieval. This section dives into the techniques that make this possible.

<a id='part3-1'></a>
## Logical Routing

![Logical Routing (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:875/1*PK9xKW0o-72xmmLaAAozeA.png)

- 根據使用者問題，經過LLM判斷要問哪個向量資料庫，去做檢索回答

In [188]:
def RouteQuery(question):
    system_prompt = """You are an expert at routing a user question to the appropriate data source.
                    Based on the programming language the question is referring to, route it to the relevant data source.
                    You must return a JSON object with a single key 'datasource'.
                    The value must be one of: ["python_docs", "js_docs", "golang_docs"].
                    Example Output: {"datasource": "python_docs"}"""

    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    content = response.choices[0].message.content
    return json.loads(content)

# from typing import Literal
# from langchain_core.pydantic_v1 import BaseModel, Field

# # Define the data model for our router's output
# class RouteQuery(BaseModel):
#     """A data model to route a user query to the most relevant datasource."""

#     # The 'datasource' field must be one of the three specified literal strings.
#     # This enforces a strict set of choices for the LLM.
#     datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
#         ...,  # The '...' indicates that this field is required.
#         description="Given a user question, choose which datasource would be most relevant for answering their question.",
#     )
# from langchain_openai import ChatOpenAI

# # Initialize our LLM
# llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

# # Create a new LLM instance that is "structured" to output our Pydantic model
# structured_llm = llm.with_structured_output(RouteQuery)

# # The system prompt provides the core instruction for the LLM's task.
# system = """You are an expert at routing a user question to the appropriate data source.

# Based on the programming language the question is referring to, route it to the relevant data source."""

# # The full prompt template combines the system message and the user's question.
# prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system", system),
#         ("human", "{question}"),
#     ]
# )

# # Define the complete router chain
# router = prompt | structured_llm


In [189]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = RouteQuery(question)
print(result)


# question = """Why doesn't the following code work:

# from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
# prompt.invoke("french")
# """

# # Invoke the router and check the result
# result = router.invoke({"question": question})

# print(result)

2026-01-26 22:40:42,372 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


{'datasource': 'python_docs'}


In [190]:
def choose_route(result):
    datasource = result.get("datasource", "").lower()
    if "python_docs" in datasource:
        # In a real app, this would be a complete RAG chain for Python docs
        return "chain for python_docs"
    elif "js_docs" in datasource:
        # This would be the chain for JavaScript docs
        return "chain for js_docs"
    else:
        # And this for Go docs
        return "chain for golang_docs"
    

router_result = RouteQuery(question)
final_destination = choose_route(router_result)
print(final_destination)

# from langchain_core.runnables import RunnableLambda

# def choose_route(result):
#     """A function to determine the downstream logic based on the router's output."""
#     if "python_docs" in result.datasource.lower():
#         # In a real app, this would be a complete RAG chain for Python docs
#         return "chain for python_docs"
#     elif "js_docs" in result.datasource.lower():
#         # This would be the chain for JavaScript docs
#         return "chain for js_docs"
#     else:
#         # And this for Go docs
#         return "chain for golang_docs"

# # The full chain now includes the routing and branching logic
# full_chain = router | RunnableLambda(choose_route)

# # Let's run the full chain
# final_destination = full_chain.invoke({"question": question})

# print(final_destination)

2026-01-26 22:40:44,926 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


chain for python_docs


<a id='part3-2'></a>
## Semantic Routing

Logical routing works perfectly when you have clearly defined categories. But what if you want to route based on the *style* or *domain* of a question? For example, you might want to answer physics questions with a serious, academic tone and math questions with a step-by-step, pedagogical approach. This is where **Semantic Routing** comes in.

![Semantic Routing (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*mzz-ncmrzdwQU37GFgPeTw.png)

依問題去選擇要使用哪個專家指令去回答
- 使用者問題轉成向量化(embedding)
- 提供專家指令(數學家跟科學家)各轉向量化，在計算與問題的向量差距，給LLM分析
- LLM選擇要使用哪個專家指令去做
- 回答答案

In [191]:
import numpy as np
from openai import OpenAI

GPT_MODEL = "gpt-oss:20b"
EMBEDDING_MODEL = "nomic-embed-text" 
client = OpenAI(base_url="http://120.113.70.238:11434/v1",api_key="ollama")

In [192]:
def get_embedding(text): 
    text = text.replace("\n", " ")
    response = client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return response.data[0].embedding

def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# from langchain.utils.math import cosine_similarity
# from langchain_openai import OpenAIEmbeddings

# # Initialize the embedding model
# embeddings = OpenAIEmbeddings()

# # Store our templates and their embeddings for comparison
# prompt_templates = [physics_template, math_template]
# prompt_embeddings = embeddings.embed_documents(prompt_templates)

# def prompt_router(input):
#     """A function to route the input query to the most similar prompt template."""
#     # 1. Embed the incoming user query
#     query_embedding = embeddings.embed_query(input["query"])
    
#     # 2. Compute the cosine similarity between the query and all prompt templates
#     similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    
#     # 3. Find the index of the most similar prompt
#     most_similar_index = similarity.argmax()
    
#     # 4. Select the most similar prompt template
#     chosen_prompt = prompt_templates[most_similar_index]
    
#     print(f"DEBUG: Using {'MATH' if most_similar_index == 1 else 'PHYSICS'} template.")
    
#     # 5. Return the chosen prompt object
#     return PromptTemplate.from_template(chosen_prompt)

In [195]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

prompt_templates = [physics_template, math_template]
prompt_names = ["Physics Professor", "Mathematician"]

prompt_embeddings = []

for text in prompt_templates:
    clean_text = text.replace("{query}", "") 
    emb = get_embedding(clean_text)
    prompt_embeddings.append(emb)


# from langchain_core.prompts import PromptTemplate

# # A prompt for a physics expert
# physics_template = """You are a very smart physics professor. \
# You are great at answering questions about physics in a concise and easy to understand manner. \
# When you don't know the answer to a question you admit that you don't know.

# Here is a question:
# {query}"""

# # A prompt for a math expert
# math_template = """You are a very good mathematician. You are great at answering math questions. \
# You are so good because you are able to break down hard problems into their component parts, \
# answer the component parts, and then put them together to answer the broader question.

# Here is a question:
# {query}"""

2026-01-26 23:10:57,765 INFO HTTP Request: POST http://120.113.70.238:11434/v1/embeddings "HTTP/1.1 200 OK"
2026-01-26 23:10:57,813 INFO HTTP Request: POST http://120.113.70.238:11434/v1/embeddings "HTTP/1.1 200 OK"


In [ ]:
def prompt_router(input_query):
    query_embedding = get_embedding(input_query)
   
    similarities = []
    for p_emb in prompt_embeddings:
        score = cosine_similarity(query_embedding, p_emb)
        similarities.append(score)

    best_index = np.argmax(similarities)
    return best_index

def main(query):
    best_index = prompt_router(query)
    selected_template = prompt_templates[best_index]
    selected_name = prompt_names[best_index]
    
    print(f"DEBUG: Using {selected_name}")
    final_prompt = selected_template.format(query=query)
    
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    return response.choices[0].message.content


result1 = main("What's a black hole")
# print(result1)
display(Markdown(result1))
# # The final chain that combines the router with the LLM
# chain = (
#     {"query": RunnablePassthrough()}
#     | RunnableLambda(prompt_router)  # Dynamically select the prompt
#     | ChatOpenAI()
#     | StrOutputParser()
# )

# # Ask a physics question
# print(chain.invoke("What's a black hole"))

2026-01-26 23:10:59,784 INFO HTTP Request: POST http://120.113.70.238:11434/v1/embeddings "HTTP/1.1 200 OK"


DEBUG: Using Physics Professor


2026-01-26 23:11:09,716 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


A **black hole** is a region of space where gravity is so strong that nothing—not even light—can escape once it crosses a certain boundary called the **event horizon**.

### How it forms
- **Stellar‑mass black holes**: When a massive star runs out of fuel, its core collapses under its own gravity. If the core’s mass exceeds about 3 solar masses, nothing can stop the collapse, and a black hole forms.
- **Supermassive black holes**: At the centers of most galaxies, including our Milky Way, lie black holes millions to billions of times the Sun’s mass. Their exact origins are still being studied, but they likely grew from smaller seeds by accreting gas and merging with other black holes.

### Key features
| Feature | What it means |
|---------|---------------|
| **Event horizon** | The “point of no return.” Anything that crosses it cannot come back. |
| **Singularity** | The core where density becomes infinite (in classical theory). In reality, quantum gravity will change this picture. |
| **Mass, spin, charge** | These are the only “observable” properties of a black hole (the “no‑hair” theorem). Mass determines the size of the horizon; spin twists spacetime; charge is usually negligible in astrophysical black holes. |

### Why we know they exist
- **Gravitational waves**: LIGO/Virgo have detected ripples from black‑hole mergers.
- **X‑ray binaries**: Matter falling into a black hole emits X‑rays; the compact object’s mass can be inferred.
- **Event Horizon Telescope**: In 2019, the first image of a black‑hole shadow (M87*) was released, showing the silhouette of the event horizon against background light.

### In short
A black hole is a spacetime region where gravity is so intense that escape velocity exceeds the speed of light. It is defined by its mass, spin, and (usually negligible) charge, and its presence is inferred from the effects on nearby matter and the gravitational waves it emits.

<a id='part3-3'></a>
## Query Structuring
把人話翻成「有條件的搜尋指令」，讓檢索更準。

把使用者用自然語言問的問題，拆解成兩部分：一部分是「要找什麼內容」（拿去做向量搜尋），另一部分是「必須符合的條件」（用 metadata 做過濾）。查詢時先用條件把不相關的資料篩掉，再在剩下的資料中做語意比對。
結構化查詢可以傳遞給支援元資料過濾的向量存儲，從而實現極其精確和高效的檢索，遠遠超越了簡單的語義搜尋。

##### 把 YouTube 原始資料 → 轉成「之後資料庫可以拿來查的欄位」
Query Structuring 之後「可以下條件的地方

In [ ]:
import yt_dlp
from datetime import datetime
from pprint import pprint
url = "https://www.youtube.com/watch?v=pbAd8O1Lvm4"

ydl_opts = {'quiet': True,'skip_download': True,'no_warnings': True}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

raw_date = info.get('upload_date')
formatted_date = str(datetime.strptime(raw_date, '%Y%m%d')) if raw_date else "Unknown"

metadata = {
    'source': info.get('id'),
    'title': info.get('title'),
    'description': info.get('description', 'Unknown'),
    'view_count': info.get('view_count'),
    'thumbnail_url': info.get('thumbnail'),
    'publish_date': formatted_date,
    'length': info.get('duration'),
    'author': info.get('uploader')
}

# print(metadata)
pprint(metadata, sort_dicts=False)

{'source': 'pbAd8O1Lvm4',
 'title': 'Self-reflective RAG with LangGraph: Self-RAG and CRAG',
 'description': 'Self-reflection can greatly enhance RAG, enabling correction '
                'of poor quality retrieval or generations. Several recent RAG '
                'papers focus on this theme, but implementing the ideas can be '
                'tricky. Here, we show that LangGraph can be easily used for '
                '"flow engineering" of self-reflective RAG pipelines. We '
                'provide cookbooks for implementing ideas from two interesting '
                'papers, Self-RAG and C-RAG.\n'
                '\n'
                'Code:\n'
                'https://github.com/langchain-ai/langgraph/tree/main/examples/rag',
 'view_count': 37050,
 'thumbnail_url': 'https://i.ytimg.com/vi/pbAd8O1Lvm4/maxresdefault.jpg',
 'publish_date': '2024-02-07 00:00:00',
 'length': 1058,
 'author': 'LangChain'}


##### 把「使用者的問題」轉成「結構化查詢」

In [49]:
def query_analyzer(question):
    author_system_prompt = """You are an expert at converting user questions into database queries. \
                            You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
                            Given a question, return a database query optimized to retrieve the most relevant results.

                            If there are acronyms or words you are not familiar with, do not try to rephrase them."""

    schema_instruction = """
                        Output Format:
                        You must return a JSON object with the following fields (all are optional except content_search and title_search):
                        {
                            "content_search": "string (similarity search query for transcripts)",
                            "title_search": "string (search query for titles)",
                            "min_view_count": "integer or null",
                            "max_view_count": "integer or null",
                            "earliest_publish_date": "string (YYYY-MM-DD) or null",
                            "latest_publish_date": "string (YYYY-MM-DD) or null",
                            "min_length_sec": "integer or null",
                            "max_length_sec": "integer or null"
                        }
                        """

    full_system_prompt = author_system_prompt + schema_instruction
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": full_system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0, 
        response_format={"type": "json_object"}
    )
    content = response.choices[0].message.content
    return json.loads(content)


def pretty_print(data):
    for key, value in data.items():
        if value is not None:
            print(f"{key}: {value}")


# import datetime
# from typing import Optional

# class TutorialSearch(BaseModel):
#     """A data model for searching over a database of tutorial videos."""

#     # The main query for a similarity search over the video's transcript.
#     content_search: str = Field(..., description="Similarity search query applied to video transcripts.")
    
#     # A more succinct query for searching just the video's title.
#     title_search: str = Field(..., description="Alternate version of the content search query to apply to video titles.")
    
#     # Optional metadata filters
#     min_view_count: Optional[int] = Field(None, description="Minimum view count filter, inclusive.")
#     max_view_count: Optional[int] = Field(None, description="Maximum view count filter, exclusive.")
#     earliest_publish_date: Optional[datetime.date] = Field(None, description="Earliest publish date filter, inclusive.")
#     latest_publish_date: Optional[datetime.date] = Field(None, description="Latest publish date filter, exclusive.")
#     min_length_sec: Optional[int] = Field(None, description="Minimum video length in seconds, inclusive.")
#     max_length_sec: Optional[int] = Field(None, description="Maximum video length in seconds, exclusive.")

#     def pretty_print(self) -> None:
#         """A helper function to print the populated fields of the model."""
#         for field in self.__fields__:
#             if getattr(self, field) is not None:
#                 print(f"{field}: {getattr(self, field)}")

# # System prompt for the query analyzer
# system = """You are an expert at converting user questions into database queries. \
# You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
# Given a question, return a database query optimized to retrieve the most relevant results.

# If there are acronyms or words you are not familiar with, do not try to rephrase them."""

# prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{question}")])
# structured_llm = llm.with_structured_output(TutorialSearch)

# # The final query analyzer chain
# query_analyzer = prompt | structured_llm

##### 只有「主題」沒有年份、長度等資訊，填寫結構化資料只有主題

In [ ]:
result1 = query_analyzer("rag from scratch")
pretty_print(result1)

# # Test 1: A simple query
# query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()

2026-01-25 22:27:35,775 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


content_search: rag from scratch
title_search: RAG from scratch


##### 句子裡有「條件」，年份要在2023年，結構化資訊就會將日期範圍2023-01-01~2023-12-31

In [ ]:
result2 = query_analyzer("videos on chat langchain published in 2023")
pretty_print(result2)

# # Test 2: A query with a date filter
# query_analyzer.invoke(
#     {"question": "videos on chat langchain published in 2023"}
# ).pretty_print()

2026-01-25 22:27:38,323 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


title_search: chat langchain
earliest_publish_date: 2023-01-01
latest_publish_date: 2023-12-31


##### 條件是「隱含的」，像是五分鐘或始很短的影片轉成結構化資訊的時間資訊

In [ ]:
result_3 = query_analyzer("how to use multi-modal models in an agent, only videos under 5 minutes")
pretty_print(result_3)

# # Test 3: A query with a length filter
# query_analyzer.invoke(
#     {
#         "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
#     }
# ).pretty_print()

2026-01-25 22:27:42,659 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


content_search: multi-modal models agent
title_search: multi-modal models agent
max_length_sec: 300


<a id='part4'></a>
# Advanced Indexing Strategies

So far, our approach to indexing has been straightforward: split documents into chunks and embed them. This works, but it has a fundamental limitation.

Small, focused chunks are great for retrieval accuracy (they contain less noise), but they often lack the broader context needed for the LLM to generate a comprehensive answer.

![Indexing Strategies (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*PrdpYBmw3-ln5AaZLjyUaw.png)

Conversely, large chunks provide great context but perform poorly in retrieval because their core meaning gets diluted.

> This is the classic “chunk size” dilemma. How can we get the best of both worlds?

The answer lies in more advanced indexing strategies that separate the document representation used for *retrieval* from the one used for *generation*. Let’s dive in.

<a id='part4-1'></a>
## Multi-Representation Indexing
父文檔檢索 (Parent Document Retrieval)

![Multi Representation Indexing (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*1TbTDTSvgVbpKxSW7feMng.png)

搜尋用「摘要」，回傳用「全文」。

##### 摘要
- 只保留 **大意與重點**
- 沒有多餘細節干擾
- 適合 **語意搜尋（Embedding / Vector Search）**
- 搜尋結果更精準

##### 全文
- 包含完整背景、細節、例子
- 是 **LLM 回答問題真正需要的素材**
- 適合拿來「讀」與「回答問題」

> **搜尋 ≠ 回答**  
> 所以使用的資料型態不應該一樣。

##### 建立資料

對每一段全文內容：
1. 產生一份 **摘要（Summary）**
2. 保留原本的 **全文（Original Document）**
3. 產生一個共同識別碼 `doc_id`

---

##### 資料儲存方式

```text
# 向量資料庫 (Vector Store)
doc_id: 42
embedding: Summa

# 一般資料庫 (Byte Store / Document Store)
doc_id: 42
content: Original Document（全文）

In [198]:
import requests
from bs4 import BeautifulSoup

def load_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    text_content = soup.get_text(separator="\n", strip=True)
    return {
        "source": url,
        "content": text_content,
        "title": soup.title.string if soup.title else "No Title"
    }
   
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2024-02-05-human-data-quality/"
]

docs = []
for url in urls:
    doc = load_url(url)
    if doc:
        docs.append(doc)

print(f"Loaded {len(docs)} documents.")


# from langchain_community.document_loaders import WebBaseLoader

# # Load two different blog posts to create a more diverse knowledge base
# loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
# docs = loader.load()

# loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
# docs.extend(loader.load())
# print(f"Loaded {len(docs)} documents.")

Loaded 2 documents.


In [200]:
def generate_summary(text):
    prompt = f"Summarize the following document:\n\n{text[:12000]}"

    response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
    temperature=0 
    )
    return response.choices[0].message.content


summaries = []
for i, doc in enumerate(docs):
    summary = generate_summary(doc['content'])
    summaries.append(summary)

# print(summaries[0])
display(Markdown(summaries[0]))

# import uuid
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import ChatPromptTemplate

# # The chain for generating summaries
# summary_chain = (
#     # Extract the page_content from the document object
#     {"doc": lambda x: x.page_content}
#     # Pipe it into a prompt template
#     | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
#     # Use an LLM to generate the summary
#     | ChatOpenAI(model="gpt-3.5-turbo", max_retries=0)
#     # Parse the output into a string
#     | StrOutputParser()
# )

# # Use .batch() to run the summarization in parallel for efficiency
# summaries = summary_chain.batch(docs, {"max_concurrency": 5})

# # Let's inspect the first summary
# print(summaries[0])

2026-01-27 00:17:12,440 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-27 00:17:25,679 INFO HTTP Request: POST http://120.113.70.238:11434/v1/chat/completions "HTTP/1.1 200 OK"


**LLM‑Powered Autonomous Agents – Key Takeaways**

| Section | Core Ideas |
|---------|------------|
| **Agent System Overview** | An LLM acts as the “brain” of an autonomous agent, supported by three main modules: <br>• **Planning** – breaking tasks into sub‑goals and refining actions.<br>• **Memory** – short‑term (in‑context) and long‑term (vector‑store + fast retrieval).<br>• **Tool Use** – calling external APIs for up‑to‑date data, code execution, proprietary sources, etc. |
| **Planning** | • **Task Decomposition** – use *Chain‑of‑Thought* (CoT) or *Tree‑of‑Thoughts* to split a problem into steps and explore multiple reasoning paths.<br>• **LLM+P** – translate a problem into PDDL, let a classical planner generate a plan, then translate back to natural language.<br>• **Self‑Reflection** – iterative improvement of past actions. <br>  – *ReAct* interleaves reasoning (“Thought”) with actions and observations.<br>  – *Reflexion* adds a heuristic to detect hallucination or inefficiency and optionally resets the episode.<br>  – *Chain of Hindsight* (CoH) trains the model on a sequence of progressively better outputs with human feedback.<br>  – *Algorithm Distillation* (AD) feeds multi‑episode RL histories into the LLM so it learns an in‑context RL policy. |
| **Memory** | • **Short‑Term / Working Memory** – in‑context prompt engineering; limited by the LLM’s context window.<br>• **Long‑Term Memory** – external vector store (e.g., FAISS) with *Maximum Inner Product Search* (MIPS) for fast retrieval of relevant facts or past experiences.<br>• Human‑brain analogies: sensory, short‑term, explicit/implicit long‑term memory. |
| **Tool Use** | Agents learn to call APIs (search, code execution, proprietary data) to supplement knowledge that is not in the model weights. |
| **Case Studies & Proof‑of‑Concepts** | • Scientific Discovery Agent – uses LLM to design experiments and interpret results.<br>• Generative Agents Simulation – agents that can simulate human‑like interactions.<br>• AutoGPT, GPT‑Engineer, BabyAGI – early demos illustrating the architecture. |
| **Challenges** | • Scaling context windows for richer memory.<br>• Avoiding hallucinations and inefficient planning.<br>• Integrating external planners or tools reliably.<br>• Balancing offline vs. online learning (e.g., AD vs. RL²). |
| **References** | The document cites key papers: Wei et al. (CoT), Yao et al. (Tree‑of‑Thoughts, ReAct), Liu et al. (CoH, LLM+P), Laskin et al. (Algorithm Distillation), Shinn & Labash (Reflexion), and classic works on human memory. |

**Bottom line:**  
LLM‑powered autonomous agents combine a language model’s reasoning ability with structured planning, reflective learning, memory retrieval, and tool‑calling to tackle complex, real‑world tasks. The field is rapidly evolving, with many research directions focused on improving self‑reflection, memory scalability, and integration with external planners and APIs.

In [201]:
client = weaviate.connect_to_local()

full_docs = client.collections.create(
    name="FullDocuments",
    properties=[
        Property(name="content", data_type=DataType.TEXT),
        Property(name="title", data_type=DataType.TEXT),
    ]
)

# 4. 建立 [摘要庫] (存摘要 + 向量搜尋)
summaries_col = client.collections.create(
    name="Summaries",
    properties=[
        Property(name="summary_text", data_type=DataType.TEXT),
        Property(name="parent_doc_id", data_type=DataType.TEXT), 
    ],
    # 設定 Ollama 向量化 (針對摘要做 Embedding)
    vectorizer_config=Configure.Vectorizer.text2vec_ollama(
        model="nomic-embed-text",
        api_endpoint="http://120.113.70.238:11434"
    ),
    # 設定生成模型 (為了之後如果有 RAG 需求)
    generative_config=Configure.Generative.ollama(
        model="gpt-oss:20b",
        api_endpoint="http://120.113.70.238:11434"
    )
)

2026-01-27 00:17:39,185 INFO HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-01-27 00:17:39,395 INFO HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-01-27 00:17:39,749 INFO HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
2026-01-27 00:17:40,029 INFO HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"
d:\RAG\rag-ecosystem\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(
2026-01-27 00:17:40,195 INFO HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"


In [ ]:
import uuid
full_docs_col = client.collections.get("FullDocuments")
summaries_col = client.collections.get("Summaries")

for i, (original_doc, summary_text) in enumerate(zip(docs, summaries)):
    parent_uuid = str(uuid.uuid4())
    
    # 2. 存入 [父文件庫] (FullDocuments)
    full_docs_col.data.insert(
        uuid=parent_uuid,
        properties={
            "content": original_doc['content'],
            "title": original_doc['title']
        }
    )
    
    # 3. 存入 [摘要庫] (Summaries)
    summaries_col.data.insert(
        properties={
            "summary_text": summary_text,
            "parent_doc_id": parent_uuid 
        }
    )
    
    print(f"已連結: 摘要 -> 父文件 ID: {parent_uuid}")

print("資料寫入完成！")


# from langchain.storage import InMemoryByteStore
# from langchain.retrievers.multi_vector import MultiVectorRetriever
# from langchain_core.documents import Document

# # The vectorstore to index the summary embeddings
# vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())

# # The storage layer for the parent documents
# store = InMemoryByteStore()
# id_key = "doc_id" # This key will link summaries to their parent documents

# # The retriever that orchestrates the whole process
# retriever = MultiVectorRetriever(
#     vectorstore=vectorstore,
#     byte_store=store,
#     id_key=id_key,
# )

# # Generate unique IDs for each of our original documents
# doc_ids = [str(uuid.uuid4()) for _ in docs]

# # Create new Document objects for the summaries, adding the 'doc_id' to their metadata
# summary_docs = [
#     Document(page_content=s, metadata={id_key: doc_ids[i]})
#     for i, s in enumerate(summaries)
# ]

# # Add the summaries to the vectorstore
# retriever.vectorstore.add_documents(summary_docs)

# # Add the original documents to the docstore, linking them by the same IDs
# retriever.docstore.mset(list(zip(doc_ids, docs)))

d:\RAG\rag-ecosystem\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\user\AppData\Local\Temp\ipykernel_10744\213181609.py:2: ResourceWarning: unclosed <socket.socket fd=3192, family=23, type=1, proto=0, laddr=('::1', 54155, 0, 0), raddr=('::1', 8080, 0, 0)>
  full_docs_col = client.collections.get("FullDocuments")
2026-01-27 00:17:43,894 INFO HTTP Request: POST http://localhost:8080/v1/objects "HTTP/1.1 200 OK"
2026-01-27 00:17:44,718 INFO HTTP Request: POST http://localhost:8080/v1/objects "HTTP/1.1 200 OK"
2026-01-27 00:17:44,741 INFO HTTP Request: POST http://localhost:8080/v1/objects "HTTP/1.1 200 OK"
2026-01-27 00:17:44,868 INFO HTTP Request: POST http://localhost:8080/v1/objects "HTTP/1.1 200 OK"


已連結: 摘要 -> 父文件 ID: 343e1131-4084-4cd6-a273-927d347374d0
已連結: 摘要 -> 父文件 ID: a7d977d5-3fa0-4da6-b945-98d135d5e37b
資料寫入完成！


In [ ]:
response = summaries_col.query.near_text(query="Memory in agents",limit=1)

hit = response.objects[0]
# print("--- Result from searching summaries ---")
# display(Markdown(hit.properties["summary_text"]))
# print("\n--- Metadata showing the link to the parent document ---")
print(f"parent_doc_id: {hit.properties['parent_doc_id']}")


parent_doc_id: 343e1131-4084-4cd6-a273-927d347374d0


In [206]:
def get_relevant_documents(query):
    response = summaries_col.query.near_text(
        query=query,
        limit=1
    )

    hit = response.objects[0]
    parent_id = hit.properties["parent_doc_id"]

    parent_obj = full_docs_col.query.fetch_object_by_id(parent_id)
    full_content = parent_obj.properties["content"]
    return full_content

full_doc_content = get_relevant_documents("Memory in agents")
print("\n--- The full document retrieved by our Weaviate Retriever ---")
print(full_doc_content[0:500])


# # Let the full retriever do its job
# retrieved_docs = retriever.get_relevant_documents(query, n_results=1)

# # Print the beginning of the retrieved full document
# print("\n--- The full document retrieved by the MultiVectorRetriever ---")
# print(retrieved_docs[0].page_content[0:500])


--- The full document retrieved by our Weaviate Retriever ---
LLM Powered Autonomous Agents | Lil'Log
Lil'Log
|
Posts
Archive
Search
Tags
FAQ
LLM Powered Autonomous Agents
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng
Table of Contents
Agent System Overview
Component One: Planning
Task Decomposition
Self-Reflection
Component Two: Memory
Types of Memory
Maximum Inner Product Search (MIPS)
Component Three: Tool Use
Case Studies
Scientific Discovery Agent
Generative Agents Simulation
Proof-of-Concept Examples
Challenges
Citati


<a id='part4-2'></a>
## Hierarchical Indexing (RAPTOR) Knowledge Tree

**The Theory:** RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval) takes the multi-representation idea a step further. Instead of just one layer of summaries, RAPTOR builds a multi-level tree of summaries. It starts by clustering small document chunks. It then summarizes each cluster.

![RAPTOR (from LangChain Docs)](https://miro.medium.com/v2/resize:fit:875/1*95v0K13O2rvsAYJ96ldhew.png)

Then, it takes these summaries, clusters *them*, and summarizes the new clusters. This process repeats, creating a hierarchy of knowledge from fine-grained details to high-level concepts. When you query, you can search at different levels of this tree, allowing for retrieval that can be as specific or as general as needed.

This is a more advanced technique, and while we won’t implement the full algorithm here, you can find a deep dive and complete code in the [RAPTOR Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/RAPTOR.ipynb). It represents the cutting edge of structured indexing.

<a id='part4-3'></a>
## Token-Level Precision (ColBERT)

**The Theory:** Standard embedding models create a single vector for an entire chunk of text (this is called a “bag-of-words” approach). This can lose a lot of nuance.

![Specialized embeddings (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:875/1*VL6Ny9Z8S9kRqgYyFhhdsA.png)

> **ColBERT (Contextualized Late Interaction over BERT)** offers a more granular approach. It generates a separate, context-aware embedding for *every single token* in the document.

When you make a query, ColBERT also embeds every token in your query. Then, instead of comparing one document vector to one query vector, it finds the maximum similarity between each query token and *any* document token.

This “late interaction” allows for a much finer-grained understanding of relevance, excelling at keyword-style searches.

We can easily use ColBERT through the `RAGatouille` library.

Now, let’s index a Wikipedia page using ColBERT’s unique token-level approach.

In [60]:
!uv add sentence-transformers

Resolved 103 packages in 827ms
Prepared 1 package in 237ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 19 packages in 8.30s
 + filelock==3.20.3
 + fsspec==2026.1.0
 + huggingface-hub==0.36.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + regex==2026.1.15
 + safetensors==0.7.0
 + scikit-learn==1.8.0
 + scipy==1.17.0
 + sentence-transformers==5.2.0
 + setuptools==80.10.1
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.2
 + torch==2.10.0
 + transformers==4.57.6
d:\RAG\rag-ecosystem\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
d:\RAG\rag-ecosystem\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unc

In [130]:
from sentence_transformers import CrossEncoder
RAG = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# from ragatouille import RAGPretrainedModel
# # Load a pre-trained ColBERT model
# RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

2026-01-25 23:41:11,009 INFO Use pytorch device: cpu


In [131]:
import requests
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_core.runnables import Runnable
from typing import List

def get_wikipedia_page(title: str):
    """A helper function to retrieve content from Wikipedia."""
    # Wikipedia API endpoint and parameters
    URL = "https://en.wikipedia.org/w/api.php"
    params = { "action": "query", "format": "json", "titles": title, "prop": "extracts", "explaintext": True }
    headers = {"User-Agent": "MyRAGApp/1.0"}
    response = requests.get(URL, params=params, headers=headers)
    data = response.json()
    page = next(iter(data["query"]["pages"].values()))
    return page.get("extract")

full_document = get_wikipedia_page("Hayao_Miyazaki")

# Manually chunk the document (simple split into sentences or paragraphs; adjust as needed)
def chunk_text(text: str, max_length: int = 180) -> List[str]:
    sentences = text.split('. ')  # Basic sentence splitting
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_length:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (". " if current_chunk else "") + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

chunks = chunk_text(full_document)

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = np.array(embed_model.encode(chunks))
def search(query: str, k: int = 10) -> List[dict]:
    query_emb = embed_model.encode(query)
    # Compute cosine similarities
    dot_products = np.dot(chunk_embeddings, query_emb)
    norms = np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    similarities = dot_products / norms
    top_indices = np.argsort(similarities)[::-1][:k]
    results = [{"content": chunks[i], "score": similarities[i]} for i in top_indices]
    return results

def rerank(query: str, results: List[dict]) -> List[dict]:
    pairs = [[query, res["content"]] for res in results]
    rerank_scores = RAG.predict(pairs)
    for i, res in enumerate(results):
        res["rerank_score"] = rerank_scores[i]
    results = sorted(results, key=lambda x: x["rerank_score"], reverse=True)
    return results

class CustomRetriever(Runnable):
    def invoke(self, query: str) -> List[Document]:
        # Retrieve top-k candidates with dense search
        candidates = search(query, k=10)
        # Rerank to get the best ones
        reranked = rerank(query, candidates)
        # Return top-3 as LangChain Documents
        docs = [Document(page_content=res["content"]) for res in reranked[:3]]
        return docs

colbert_retriever = CustomRetriever()



# import requests
# def get_wikipedia_page(title: str):
#     """A helper function to retrieve content from Wikipedia."""
#     # Wikipedia API endpoint and parameters
#     URL = "https://en.wikipedia.org/w/api.php"
#     params = { "action": "query", "format": "json", "titles": title, "prop": "extracts", "explaintext": True }
#     headers = {"User-Agent": "MyRAGApp/1.0"}
#     response = requests.get(URL, params=params, headers=headers)
#     data = response.json()
#     page = next(iter(data["query"]["pages"].values()))
#     return page.get("extract")

# full_document = get_wikipedia_page("Hayao_Miyazaki")

# # Index the document with RAGatouille. It handles the chunking and token-level embedding internally.
# RAG.index(
#     collection=[full_document],
#     index_name="Miyazaki-ColBERT",
#     max_document_length=180,
#     split_documents=True,
# )

2026-01-25 23:41:16,543 INFO Use pytorch device_name: cpu
2026-01-25 23:41:16,543 INFO Load pretrained SentenceTransformer: all-MiniLM-L6-v2
Batches: 100%|██████████| 11/11 [00:01<00:00,  6.02it/s]


In [133]:
results = search(query="What animation studio did Miyazaki found?", k=3)
print(results)

Batches: 100%|██████████| 1/1 [00:00<00:00, 100.82it/s]

[{'content': 'Miyazaki produced and financed the film, and provided several animated sequences', 'score': 0.7450253}, {'content': 'Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the most accomplished filmmakers in the history of animation.\nBorn in Tokyo City, Miyazaki expressed interest in manga and animation from an early age', 'score': 0.7291059}, {'content': "It was Miyazaki's final television work", 'score': 0.7273969}]


In [132]:
retrieved_docs = colbert_retriever.invoke("What animation studio did Miyazaki found?")
print(retrieved_docs[0].page_content)

Batches: 100%|██████████| 1/1 [00:00<00:00, 14.89it/s]

Studio Ghibli became a subsidiary of Nippon Television Holdings in October 2023, with Miyazaki as its honorary chairman.


== Views ==

Miyazaki has often criticized the state of the animation industry, stating that some animators lack a foundational understanding of their subjects and do not prioritize realism


<a id='part5'></a>
# Advanced Retrieval & Generation

We have created a sophisticated RAG system with intelligent routing and advanced indexing. Now, we’ve reached the final mile: retrieval and generation. This is where we ensure the context we feed to the LLM is of the highest possible quality and that the LLM’s final answer is relevant, accurate, and grounded in that context.

![Retrieval/Generation (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*RJzBqSbw8V0LPpzYN7VFjA.png)

Even with the best indexing, our initial retrieval can still contain noise less relevant documents that slip through. And LLMs, powerful as they are, can sometimes misunderstand context or hallucinate.

This section introduces the advanced techniques that act as the final quality control layer for our pipeline.

<a id='part5-1'></a>
## Dedicated Re-ranking

Standard retrieval methods give us a ranked list of documents, but this initial ranking isn’t always perfect. **Re-ranking** is a crucial second-pass step where we take the initial set of retrieved documents and use a more sophisticated (and often more expensive) model to re-order them based on their relevance to the query.

![Dedicated Re-Ranking (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*rnQCpniADswmhbTFiCN1Gg.png)

> This ensures that the most relevant documents are placed at the very top of the context we provide to the LLM.

We have already seen one powerful re-ranking method: Reciprocal Rank Fusion (RRF) in our RAG-Fusion section. It’s a great, model-free way to combine results. But for an even more powerful approach, we can use a dedicated re-ranking model, like the one provided by Cohere.

Let’s set up a standard retriever first. We’ll use the same blog post from our previous examples.

In [50]:
# You will need to set your COHERE_API_KEY environment variable
# os.environ['COHERE_API_KEY'] = '<your-cohere-api-key>'

# Load, split, and index the document
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",))
blog_docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(blog_docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# First-pass retriever: get the top 10 potentially relevant documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

Now, we introduce the `ContextualCompressionRetriever`. This special retriever wraps our base retriever and adds a "compressor" step. Here, our compressor will be the `CohereRerank` model.

It will take the 10 documents from our base retriever and re-order them, returning only the most relevant ones.

In [51]:
# You will need to install cohere: pip install cohere
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

# Initialize the Cohere Rerank model
compressor = CohereRerank()

# Create the compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=retriever
)

# Let's test it with our query
question = "What is task decomposition for LLM agents?"
compressed_docs = compression_retriever.get_relevant_documents(question)

# Print the re-ranked documents
print("--- Re-ranked and Compressed Documents ---")
for doc in compressed_docs:
    print(f"Relevance Score: {doc.metadata['relevance_score']:.4f}")
    print(f"Content: {doc.page_content[:150]}...\n")

--- Re-ranked and Compressed Documents ---
Relevance Score: 0.9982
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.", "What are the subgoals for achieving XYZ?", (2) by using task...

Relevance Score: 0.9851
Content: Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into mult...

Relevance Score: 0.9765
Content: LLM-powered autonomous agents have been an exciting concept. They can be used for task decomposition by prompting, using task-specific instructions, or ...



The output is remarkable. The `CohereRerank` model has not only re-ordered the documents but has also assigned a `relevance_score` to each one. We can now be much more confident that the context we pass to the LLM is of the highest quality, directly leading to better, more accurate answers.

<a id='part5-2'></a>
## Self-Correction using AI Agents

What if our RAG system could check its own work before giving an answer? That’s the idea behind self-correcting RAG architectures like **CRAG (Corrective RAG)** and **Self-RAG**.

![Self Correction RAG (From Langchain blog)](https://miro.medium.com/v2/resize:fit:875/1*LpQrsvNj09aJPMhhh4fc-A.png)

These aren’t just simple chains, they are dynamic graphs (often built with LangGraph) that can reason about the quality of retrieved information and decide on a course of action.

- **CRAG:** If the retrieved documents are irrelevant or ambiguous for a given query, a CRAG system won’t just pass them to the LLM. Instead, it triggers a new, more robust web search to find better information, corrects the retrieved documents, and then proceeds with generation.
- **Self-RAG:** This approach takes it a step further. At each step, it uses an LLM to generate “reflection tokens” that critique the process. It grades the retrieved documents for relevance. If they’re not relevant, it retrieves again. Once it has good documents, it generates an answer and then grades that answer for factual consistency, ensuring it’s grounded in the source documents.

These techniques represent the state-of-the-art in building reliable, production-grade RAG. Implementing them from scratch involves building a state machine or graph. While the full implementation is extensive, you can find excellent, detailed walkthroughs here:

- [CRAG Notebook](https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb)
- [Self-RAG Notebook](https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag_mistral_nomic.ipynb)

These agentic frameworks are the key to moving beyond simple Q&A bots to creating truly robust reasoning engines.

<a id='part5-3'></a>
## Impact of Long Context

A recurring theme in RAG has been the limited context windows of LLMs. But with the rise of models boasting massive context windows (128k, 200k, or even 1 million tokens), a question arises:

![Long Context (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:6986/1*g3NCw9EzZcylHpOJMlGr8A.png)

> **Do we still need RAG?** Can we just stuff all our documents into the prompt?

The answer is nuanced. While long context models are incredibly powerful, they are not a silver bullet.

Research has shown that their performance can degrade when the crucial information is buried in the middle of a very long context (the “needle in a haystack” problem).

- **RAG Advantage:** RAG excels at *finding* the needle first and presenting only that to the LLM. It’s a precision tool.
- **Long Context’s Advantage:** Long context models are fantastic for tasks that require synthesizing information from *many different parts* of a document simultaneously, something RAG might miss.

The future is likely a hybrid approach: using RAG to perform an initial, precise retrieval of the most relevant documents and then feeding this high-quality, pre-filtered context into a long-context model for final synthesis.

For a deep dive into this topic, this presentation is an excellent resource:

- **Slides on Long Context:** [The Impact of Long Context on RAG](https://docs.google.com/presentation/d/1mJUiPBdtf58NfuSEQ7pVSEQ2Oqmek7F1i4gBwR6JDss/edit#slide=id.g26c0cb8dc66_0_0)

<a id='part6'></a>
# Manual RAG Evaluation

We have built an increasingly sophisticated RAG pipeline, layering on advanced techniques for retrieval, indexing, and generation. But a crucial question remains: **how do we prove it actually works?**

In a production environment, “it seems to work” is not enough. We need objective, repeatable metrics to measure performance, identify weaknesses, and guide improvements.

This is where evaluation comes in. It’s the science of holding our RAG system accountable. In this part, we will explore how to quantitatively measure our system’s quality by building our own evaluators from first principles.

<a id='part6-1'></a>
## The Core Metrics: What Should We Measure?

Before we dive into code, let’s define what a “good” RAG response looks like. We can break it down into a few core principles:

1. **Faithfulness:** Does the answer stick strictly to the provided context? A faithful answer does not invent information or use the LLM’s pre-trained knowledge to answer. This is the single most important metric for preventing hallucinations.
2. **Correctness:** Is the answer factually correct when compared to a “ground truth” or reference answer?
3. **Contextual Relevancy:** Was the context we retrieved actually relevant to the user’s question? This evaluates the performance of our retriever, not the generator.

Let’s explore how to measure these, starting with the most transparent method: building the evaluators ourselves.

<a id='part6-2'></a>
## Building Evaluators from Scratch with LangChain

The best way to understand evaluation is to build it. Using basic LangChain components, we can create custom chains that instruct an LLM to act as an impartial “judge”, grading our RAG system’s output based on criteria we define in a prompt. This gives us maximum control and transparency.

Let’s begin with **Correctness**. Our goal is to create a chain that compares the generated_answer to a ground_truth answer and returns a score from 0 to 1.

In [52]:
from langchain.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# We'll use a powerful LLM like gpt-4o to act as our "judge" for reliable evaluation.
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)

# Define the output schema for our evaluation score to ensure consistent, structured output.
class ResultScore(BaseModel):
    score: float = Field(..., description="The score of the result, ranging from 0 to 1 where 1 is the best possible score.")

# This prompt template clearly instructs the LLM on how to score the answer's correctness.
correctness_prompt = PromptTemplate(
    input_variables=["question", "ground_truth", "generated_answer"],
    template="""
    Question: {question}
    Ground Truth: {ground_truth}
    Generated Answer: {generated_answer}

    Evaluate the correctness of the generated answer compared to the ground truth.
    Score from 0 to 1, where 1 is perfectly correct and 0 is completely incorrect.
    
    Score:
    """
)

# We build the evaluation chain by piping the prompt to the LLM with structured output.
correctness_chain = correctness_prompt | llm.with_structured_output(ResultScore)

Now, let’s wrap this in a simple function and test it. What if the ground truth is “Paris and Madrid” but our RAG system only partially answered with “Paris”?

In [53]:
def evaluate_correctness(question, ground_truth, generated_answer):
    """A helper function to run our custom correctness evaluation chain."""
    result = correctness_chain.invoke({
        "question": question, 
        "ground_truth": ground_truth, 
        "generated_answer": generated_answer
    })
    return result.score

# Test the correctness chain with a partially correct answer.
question = "What is the capital of France and Spain?"
ground_truth = "Paris and Madrid"
generated_answer = "Paris"
score = evaluate_correctness(question, ground_truth, generated_answer)

print(f"Correctness Score: {score}")

Correctness Score: 0.5


This is a perfect result. Our judge LLM correctly reasoned that the generated answer was only half-correct and assigned an appropriate score of 0.5.

Next, let’s build an evaluator for **Faithfulness**. This is arguably more important than correctness for RAG, as it’s our primary defense against hallucination.

Here, the judge LLM must ignore whether the answer is factually correct and *only* care if the answer can be derived from the given `context`.

In [54]:
# The prompt template for faithfulness includes several examples (few-shot prompting)
# to make the instructions to the judge LLM crystal clear.
faithfulness_prompt = PromptTemplate(
    input_variables=["question","context", "generated_answer"],
    template="""
    Question: {question}
    Context: {context}
    Generated Answer: {generated_answer}

    Evaluate if the generated answer to the question can be deduced from the context.
    Score of 0 or 1, where 1 is perfectly faithful *AND CAN BE DERIVED FROM THE CONTEXT* and 0 otherwise.
    You don't mind if the answer is correct; all you care about is if the answer can be deduced from the context.
    
    Example:
    Question: What is the capital of France and Spain?
    Context: Paris is the capital of France and Madrid is the capital of Spain.
    Generated Answer: Paris
    in this case the generated answer is faithful to the context so the score should be *1*.
    
    Example:
    Question: What is 2+2?
    Context: 4.
    Generated Answer: 4.
    In this case, the context states '4', but it does not provide information to deduce the answer to 'What is 2+2?', so the score should be 0.
    """
)

# Build the faithfulness chain using the same structured LLM.
faithfulness_chain = faithfulness_prompt | llm.with_structured_output(ResultScore)

We’ve provided several examples in the prompt to guide the LLM’s reasoning, especially for tricky edge cases. Let’s test it with the “2+2” example, which is a classic test for faithfulness.

In [55]:
def evaluate_faithfulness(question, context, generated_answer):
    """A helper function to run our custom faithfulness evaluation chain."""
    result = faithfulness_chain.invoke({
        "question": question, 
        "context": context, 
        "generated_answer": generated_answer
    })
    return result.score

# Test the faithfulness chain. The answer is correct, but is it faithful?
question = "what is 3+3?"
context = "6"
generated_answer = "6"
score = evaluate_faithfulness(question, context, generated_answer)

print(f"Faithfulness Score: {score}")

Faithfulness Score: 0.0


This demonstrates the power and precision of a well-defined faithfulness metric. Even though the answer **6** is factually correct, it could not be logically deduced from the provided context “6”.

The context didn’t say **3+3 equals 6**. Our system correctly flagged this as an unfaithful answer, which is likely a hallucination where the LLM used its own pre-trained knowledge instead of the provided context.

Building these evaluators from scratch provides deep insight into what we’re measuring. However, it can be time-consuming. In the next part, we’ll see how to achieve the same results more efficiently using specialized evaluation frameworks.

<a id='part7'></a>
# Evaluation with Frameworks

In the previous part, we built our own evaluation chains from scratch. This is a fantastic way to understand the core principles of RAG metrics.

> However, for faster and more robust testing, dedicated evaluation frameworks are the way to go.

![Eval using Frameworks (Created by Fareed Khan)](https://miro.medium.com/v2/resize:fit:1250/1*uBn-2vN1Bz--NXfaeR2hyw.png)

These libraries provide pre-built, fine-tuned metrics that handle the complexity of evaluation for us, allowing us to focus on analyzing the results.

We’ll explore three popular frameworks: `deepeval`, `grouse`, and the RAG-specific powerhouse, `RAGAS`.

<a id='part7-1'></a>
## Rapid Evaluation with `deepeval`

`deepeval` is a powerful, open-source framework designed to make LLM evaluation simple and intuitive. It provides a set of well-defined metrics that can be easily applied to your RAG pipeline's outputs.

The workflow involves creating `LLMTestCase` objects and measuring them against pre-built metrics like `Correctness`, `Faithfulness`, and `ContextualRelevancy`.

In [56]:
# You will need to install deepeval: pip install deepeval
from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

# Create test cases
test_case_correctness = LLMTestCase(
    input="What is the capital of Spain?",
    expected_output="Madrid is the capital of Spain.",
    actual_output="MadriD."
)

test_case_faithfulness = LLMTestCase(
    input="what is 3+3?",
    actual_output="6",
    retrieval_context=["6"]
)

# The evaluate() function runs all test cases against all specified metrics
evaluation_results = evaluate(
    test_cases=[test_case_correctness, test_case_faithfulness],
    metrics=[GEval(name="Correctness", model="gpt-4o"), FaithfulnessMetric()]
)

print(evaluation_results)

✨ Evaluation Results ✨
-------------------------
Overall Score: 0.50
-------------------------
Metrics Summary:
- Correctness: 1.00
- Faithfulness: 0.00
-------------------------


The aggregated view from `deepeval` immediately gives us a high-level picture of our system's performance, making it easy to spot areas that need improvement.

<a id='part7-2'></a>
## Another Powerful Alternative with `grouse`

`grouse` is another excellent open-source option, offering a similar suite of metrics but with a unique focus on allowing deep customization of the "judge" prompts. This is useful for fine-tuning evaluation criteria for a specific domain.

In [57]:
# You will need to install grouse: pip install grouse-eval
from grouse import EvaluationSample, GroundedQAEvaluator

evaluator = GroundedQAEvaluator()
unfaithful_sample = EvaluationSample(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower is located at Rue Rabelais in Paris.",
    references=[
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France",
        "Gustave Eiffel died in his appartment at Rue Rabelais in Paris."
    ]
)

result = evaluator.evaluate(eval_samples=[unfaithful_sample]).evaluations[0]
print(f"Grouse Faithfulness Score (0 or 1): {result.faithfulness.faithfulness}")

Grouse Faithfulness Score (0 or 1): 0


Like `deepeval`, `grouse` effectively catches subtle errors, providing another robust tool for our evaluation toolkit.

<a id='part7-3'></a>
## Evaluation with `RAGAS`

While `deepeval` and `grouse` are great general-purpose evaluators, **RAGAS (Retrieval-Augmented Generation Assessment)** is a framework built *specifically* for evaluating RAG pipelines. It provides a comprehensive suite of metrics that measure every component of your system, from retriever to generator.

[Image of the RAGAS logo with its core metrics: Faithfulness, Answer Relevancy, Context Recall, etc.]

To use `RAGAS`, we first need to prepare our evaluation data in a specific format. It requires four key pieces of information for each test case:

- `question`: The user's input query.
- `answer`: The final answer generated by our RAG system.
- `contexts`: The list of documents retrieved by our retriever.
- `ground_truth`: The correct, reference answer.

Let’s prepare a sample dataset.

In [58]:
# 1. Prepare the evaluation data
questions = [
    "What is the name of the three-headed dog guarding the Sorcerer's Stone?",
    "Who gave Harry Potter his first broomstick?",
    "Which house did the Sorting Hat initially consider for Harry?",
]

# These would be the answers generated by our RAG pipeline
generated_answers = [
    "The three-headed dog is named Fluffy.",
    "Professor McGonagall gave Harry his first broomstick, a Nimbus 2000.",
    "The Sorting Hat strongly considered putting Harry in Slytherin.",
]

# The ground truth, or "perfect" answers
ground_truth_answers = [
    "Fluffy",
    "Professor McGonagall",
    "Slytherin",
]

# The context retrieved by our RAG system for each question
retrieved_documents = [
    ["A massive, three-headed dog was guarding a trapdoor. Hagrid mentioned its name was Fluffy."],
    ["First years are not allowed brooms, but Professor McGonagall, head of Gryffindor, made an exception for Harry."],
    ["The Sorting Hat muttered in Harry's ear, 'You could be great, you know, it's all here in your head, and Slytherin will help you on the way to greatness...'"],
]

Next, we structure this data using the Hugging Face `datasets` library, which `RAGAS` integrates with seamlessly.

In [59]:
# You will need to install ragas and datasets: pip install ragas datasets
from datasets import Dataset

# 2. Structure the data into a Hugging Face Dataset object
data_samples = {
    'question': questions,
    'answer': generated_answers,
    'contexts': retrieved_documents,
    'ground_truth': ground_truth_answers
}

dataset = Dataset.from_dict(data_samples)

Now, we can define our metrics and run the evaluation. `RAGAS` offers several powerful, RAG-specific metrics out of the box.

In [60]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_correctness,
)

# 3. Define the metrics we want to use for evaluation
metrics = [
    faithfulness,       # How factually consistent is the answer with the context? (Prevents hallucination)
    answer_relevancy,   # How relevant is the answer to the question?
    context_recall,     # Did we retrieve all the necessary context to answer the question?
    answer_correctness, # How accurate is the answer compared to the ground truth?
]

# 4. Run the evaluation
result = evaluate(
    dataset=dataset, 
    metrics=metrics
)

# 5. Display the results in a clean table format
results_df = result.to_pandas()
print(results_df)

                                             question  ... answer_correctness
0  What is the name of the three-headed dog guard...    ...           1.000000
1          Who gave Harry Potter his first broomstick?  ...           0.954321
2  Which house did the Sorting Hat initially cons...    ...           1.000000


We can see that our system is highly faithful and retrieves relevant context well (`faithfulness` and `context_recall` are perfect). The answers are also highly relevant and correct, with only minor deviations.

`RAGAS` makes it incredibly easy to run this kind of comprehensive, end-to-end evaluation, giving us the data we need to confidently deploy and improve our RAG applications.

<a id='part8'></a>
# Summarizing Everything

So, let’s sum up what we have done so far on our way to build a production-ready RAG system.

- In **Part 1**, we built a foundational RAG system from the ground up, covering the three core components: **Indexing** our data, **Retrieving** relevant context, and **Generating** a final answer.
- In **Part 2**, we moved to **Advanced Query Transformations**, using techniques like RAG-Fusion, Decomposition, and HyDE to rewrite and expand user questions for far more accurate retrieval.
- In **Part 3**, we turned our pipeline into an intelligent switchboard, adding **Routing** to direct queries to the correct data source and **Query Structuring** to leverage powerful metadata filters.
- In **Part 4**, we focused on **Advanced Indexing**, exploring strategies like Multi-Representation Indexing and token-level ColBERT to create a smarter and more efficient knowledge library.
- In **Part 5**, we polished the final output with **Advanced Retrieval** techniques like re-ranking to prioritize the best context and introduced agentic, self-correcting concepts like CRAG and Self-RAG.
- Finally, in **Parts 6 and 7**, we tackled the crucial step of **Evaluation**. We learned how to measure our system’s performance with key metrics like faithfulness and correctness, both by building evaluators from scratch and by using powerful frameworks like deepeval, grouse, and RAGAS.

> In case you enjoy this blog, feel free to **[follow me on Medium](https://medium.com/@fareedkhandev)** I only write here.